In [1]:
# ============================================================
# DAY 62/100 — ENTERPRISE AGENTIC RAG + VECTOR DB
# PART 1 — VECTOR DATABASE FOUNDATION
# ============================================================
#
# Goal:
# Build a lightweight enterprise vector database foundation
# containing:
#
# 1. Enterprise documents
# 2. Document cleaning
# 3. Chunking
# 4. Metadata extraction
# 5. Vector representation
# 6. In-memory vector index
# 7. Similarity search
# 8. Metadata filtering
# 9. Top-K retrieval
# 10. Retrieval validation
#
# CPU + storage friendly implementation.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import re
import time
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


print("Libraries loaded successfully.")


# ============================================================
# 2. PROJECT CONFIGURATION
# ============================================================

VECTOR_DB_CONFIG = {
    "project_name": "Enterprise Agentic RAG & Vector Database Intelligence Platform",
    "embedding_type": "TF-IDF Sparse Vector",
    "vector_dimension": None,
    "chunk_size": 80,
    "chunk_overlap": 15,
    "top_k": 5,
    "similarity_threshold": 0.05,
    "metric": "cosine_similarity"
}

print("\nProject configuration:")
for key, value in VECTOR_DB_CONFIG.items():
    print(f"{key}: {value}")


# ============================================================
# 3. ENTERPRISE DOCUMENT DATASET
# ============================================================
#
# Small synthetic enterprise dataset.
#
# Domains:
# - HR
# - Finance
# - Security
# - Insurance
# - Technology
#
# This avoids downloading external datasets.
# ============================================================

enterprise_documents = [
    
    {
        "document_id": "HR-001",
        "title": "Employee Leave Policy",
        "department": "HR",
        "document_type": "Policy",
        "source": "hr_leave_policy.pdf",
        "content": """
        Employees are eligible for annual leave according to their employment
        category and organizational policy. Leave requests should be submitted
        through the employee management portal before the planned absence.
        Managers are responsible for reviewing and approving leave requests.
        Emergency leave should be communicated to the reporting manager as
        soon as reasonably possible. Employees should review the official HR
        policy for applicable leave balances and eligibility requirements.
        """
    },
    
    {
        "document_id": "HR-002",
        "title": "Remote Work Policy",
        "department": "HR",
        "document_type": "Policy",
        "source": "remote_work_policy.pdf",
        "content": """
        Employees may work remotely based on organizational eligibility,
        business requirements, manager approval, and applicable security
        policies. Remote employees must use approved corporate devices,
        secure authentication mechanisms, and authorized communication
        systems. Sensitive company information must not be stored on
        unauthorized personal devices or services.
        """
    },
    
    {
        "document_id": "FIN-001",
        "title": "Expense Reimbursement Policy",
        "department": "Finance",
        "document_type": "Policy",
        "source": "expense_policy.pdf",
        "content": """
        Employees can request reimbursement for eligible business expenses.
        Expense claims should include valid receipts and supporting documents.
        Claims must be submitted through the approved expense management
        system within the required submission period. Finance teams review
        expenses according to company reimbursement rules and applicable
        approval limits.
        """
    },
    
    {
        "document_id": "FIN-002",
        "title": "Corporate Payment Card Policy",
        "department": "Finance",
        "document_type": "Policy",
        "source": "corporate_card_policy.pdf",
        "content": """
        Corporate payment cards are issued to authorized employees for
        approved business activities. Cardholders must protect card
        credentials and immediately report suspicious transactions.
        Personal purchases are prohibited unless explicitly authorized by
        company policy. Monthly statements should be reviewed and reconciled
        against business expenses.
        """
    },
    
    {
        "document_id": "SEC-001",
        "title": "Information Security Policy",
        "department": "Security",
        "document_type": "Security Policy",
        "source": "information_security_policy.pdf",
        "content": """
        Employees must protect confidential company information and follow
        approved security controls. Passwords must not be shared with other
        users. Multi-factor authentication should be enabled for supported
        applications. Security incidents, suspicious activity, and potential
        data exposure must be reported through the approved security process.
        """
    },
    
    {
        "document_id": "SEC-002",
        "title": "Vulnerability Management Policy",
        "department": "Security",
        "document_type": "Security Policy",
        "source": "vulnerability_management_policy.pdf",
        "content": """
        Security vulnerabilities should be identified, assessed, prioritized,
        and remediated according to organizational risk management procedures.
        Critical vulnerabilities require expedited remediation. Vulnerability
        priority may depend on severity, exploitability, asset exposure,
        business impact, and whether the affected component is actively used.
        """
    },
    
    {
        "document_id": "INS-001",
        "title": "Group Insurance Underwriting Policy",
        "department": "Insurance",
        "document_type": "Underwriting Policy",
        "source": "group_underwriting_policy.pdf",
        "content": """
        Group insurance underwriting evaluates applicant information,
        coverage requirements, eligibility criteria, risk factors, and
        applicable underwriting guidelines. Underwriters should review
        submitted documentation and identify missing or inconsistent
        information before completing risk assessment.
        """
    },
    
    {
        "document_id": "INS-002",
        "title": "Insurance Claims Processing",
        "department": "Insurance",
        "document_type": "Claims Policy",
        "source": "claims_processing_policy.pdf",
        "content": """
        Insurance claims should be validated against policy information,
        claimant information, supporting documentation, coverage conditions,
        and applicable claims rules. Claims with incomplete information
        should be identified for additional review before final processing.
        """
    },
    
    {
        "document_id": "TECH-001",
        "title": "API Development Standards",
        "department": "Technology",
        "document_type": "Engineering Standard",
        "source": "api_development_standards.pdf",
        "content": """
        Enterprise APIs should follow consistent naming conventions,
        request and response validation, authentication, authorization,
        structured error handling, logging, and API documentation standards.
        Production APIs should expose health checks and appropriate monitoring
        metrics.
        """
    },
    
    {
        "document_id": "TECH-002",
        "title": "Application Deployment Standard",
        "department": "Technology",
        "document_type": "Engineering Standard",
        "source": "deployment_standard.pdf",
        "content": """
        Applications should be validated before production deployment.
        Deployment pipelines should include automated testing, environment
        validation, configuration management, logging, monitoring, and
        rollback capabilities. Production releases should follow approved
        change management procedures.
        """
    }
]


documents_df = pd.DataFrame(enterprise_documents)

print("\nDataset created successfully.")
print(f"Documents: {len(documents_df)}")

display(
    documents_df[
        [
            "document_id",
            "title",
            "department",
            "document_type"
        ]
    ]
)


# ============================================================
# 4. DOCUMENT CLEANING
# ============================================================

def clean_text(text):
    """
    Basic enterprise document cleaning.
    """

    text = str(text)

    # Remove excessive whitespace
    text = re.sub(r"\s+", " ", text)

    # Normalize spaces
    text = text.strip()

    return text


documents_df["clean_content"] = documents_df["content"].apply(clean_text)

print("\nDocument cleaning completed.")

print("\nExample:")
print(documents_df.iloc[0]["clean_content"][:300])


# ============================================================
# 5. DOCUMENT CHUNKING
# ============================================================
#
# Why chunking?
#
# Large documents should not be stored/retrieved as one huge block.
#
# Instead:
#
# Document
#    ↓
# Smaller chunks
#    ↓
# Vector representation
#    ↓
# Retrieval
#
# Each chunk keeps its original metadata.
# ============================================================

def chunk_text(text, chunk_size=80, overlap=15):
    """
    Character-based lightweight chunking.

    Parameters
    ----------
    text : str
        Input document text.

    chunk_size : int
        Approximate number of words per chunk.

    overlap : int
        Number of overlapping words.
    """

    words = text.split()

    if not words:
        return []

    chunks = []

    start = 0

    while start < len(words):

        end = start + chunk_size

        chunk = " ".join(words[start:end])

        chunks.append(chunk)

        if end >= len(words):
            break

        start = end - overlap

    return chunks


chunk_records = []

for _, row in documents_df.iterrows():

    chunks = chunk_text(
        row["clean_content"],
        chunk_size=VECTOR_DB_CONFIG["chunk_size"],
        overlap=VECTOR_DB_CONFIG["chunk_overlap"]
    )

    for chunk_index, chunk in enumerate(chunks):

        chunk_records.append({
            "chunk_id": f'{row["document_id"]}-CHUNK-{chunk_index}',
            "document_id": row["document_id"],
            "title": row["title"],
            "department": row["department"],
            "document_type": row["document_type"],
            "source": row["source"],
            "chunk_index": chunk_index,
            "text": chunk
        })


chunks_df = pd.DataFrame(chunk_records)

print("\nChunking completed.")
print(f"Original documents : {len(documents_df)}")
print(f"Total chunks       : {len(chunks_df)}")

display(
    chunks_df[
        [
            "chunk_id",
            "document_id",
            "department",
            "document_type",
            "chunk_index"
        ]
    ].head(15)
)


# ============================================================
# 6. CHUNK STATISTICS
# ============================================================

chunks_df["word_count"] = chunks_df["text"].apply(
    lambda x: len(x.split())
)

print("\nChunk statistics:")

print(
    chunks_df["word_count"].describe()
)


# ============================================================
# 7. BUILD VECTOR REPRESENTATION
# ============================================================
#
# For CPU/storage efficiency we use TF-IDF.
#
# Concept:
#
# Documents
#     ↓
# TF-IDF Vectorizer
#     ↓
# Sparse vectors
#
# Query:
#
# User Query
#     ↓
# Same Vectorizer
#     ↓
# Query Vector
#
# Then:
#
# Query Vector
#      ↓
# Cosine Similarity
#      ↓
# Chunk Similarity
#
# IMPORTANT:
# TF-IDF is a lexical vector representation.
# It is NOT a dense semantic embedding model.
# ============================================================

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    ngram_range=(1, 2),
    max_features=3000
)

chunk_vectors = vectorizer.fit_transform(
    chunks_df["text"]
)

VECTOR_DB_CONFIG["vector_dimension"] = chunk_vectors.shape[1]

print("\nVector representation created.")

print(f"Number of vectors : {chunk_vectors.shape[0]}")
print(f"Vector dimension  : {chunk_vectors.shape[1]}")
print(f"Matrix shape      : {chunk_vectors.shape}")


# ============================================================
# 8. LIGHTWEIGHT VECTOR DATABASE CLASS
# ============================================================
#
# This class provides a simple vector database abstraction.
#
# Production equivalent:
#
# Chroma
# Pinecone
# Weaviate
# Qdrant
# Milvus
# Azure AI Search
# Elasticsearch / OpenSearch
#
# Our interface:
#
# add()
# search()
# metadata_filter()
# ============================================================

class LightweightVectorDB:

    def __init__(self, vectorizer, vectors, metadata_df):

        self.vectorizer = vectorizer
        self.vectors = vectors
        self.metadata_df = metadata_df.reset_index(drop=True)

        self.count = vectors.shape[0]

    def _filter_indices(self, filters=None):

        if not filters:
            return np.arange(self.count)

        mask = np.ones(self.count, dtype=bool)

        for key, value in filters.items():

            if key not in self.metadata_df.columns:
                raise ValueError(
                    f"Unknown metadata field: {key}"
                )

            mask &= (
                self.metadata_df[key].astype(str).str.lower()
                == str(value).lower()
            )

        return np.where(mask)[0]

    def search(
        self,
        query,
        top_k=5,
        filters=None,
        similarity_threshold=0.0
    ):

        # ----------------------------------------------------
        # Convert query into vector
        # ----------------------------------------------------

        query_vector = self.vectorizer.transform([query])

        # ----------------------------------------------------
        # Metadata filtering
        # ----------------------------------------------------

        candidate_indices = self._filter_indices(filters)

        if len(candidate_indices) == 0:
            return pd.DataFrame(
                columns=[
                    "chunk_id",
                    "document_id",
                    "title",
                    "department",
                    "document_type",
                    "source",
                    "chunk_index",
                    "text",
                    "similarity_score"
                ]
            )

        candidate_vectors = self.vectors[candidate_indices]

        # ----------------------------------------------------
        # Similarity calculation
        # ----------------------------------------------------

        scores = cosine_similarity(
            query_vector,
            candidate_vectors
        )[0]

        # ----------------------------------------------------
        # Ranking
        # ----------------------------------------------------

        ranking = np.argsort(scores)[::-1]

        selected_indices = []

        for position in ranking:

            score = float(scores[position])

            if score >= similarity_threshold:
                selected_indices.append(position)

            if len(selected_indices) >= top_k:
                break

        if not selected_indices:

            return pd.DataFrame(
                columns=[
                    "chunk_id",
                    "document_id",
                    "title",
                    "department",
                    "document_type",
                    "source",
                    "chunk_index",
                    "text",
                    "similarity_score"
                ]
            )

        original_indices = [
            candidate_indices[position]
            for position in selected_indices
        ]

        results = self.metadata_df.iloc[
            original_indices
        ].copy()

        results["similarity_score"] = [
            float(scores[position])
            for position in selected_indices
        ]

        results = results.sort_values(
            "similarity_score",
            ascending=False
        )

        return results.reset_index(drop=True)


# ============================================================
# 9. CREATE VECTOR DATABASE
# ============================================================

vector_db = LightweightVectorDB(
    vectorizer=vectorizer,
    vectors=chunk_vectors,
    metadata_df=chunks_df
)

print("\nVector database initialized.")

print(f"Indexed vectors: {vector_db.count}")


# ============================================================
# 10. BASIC VECTOR SEARCH
# ============================================================

query = "How should employees request leave?"

start_time = time.perf_counter()

results = vector_db.search(
    query=query,
    top_k=5,
    similarity_threshold=VECTOR_DB_CONFIG["similarity_threshold"]
)

latency_ms = (time.perf_counter() - start_time) * 1000

print("\n============================================================")
print("BASIC VECTOR SEARCH")
print("============================================================")

print(f"Query   : {query}")
print(f"Results : {len(results)}")
print(f"Latency : {latency_ms:.2f} ms")

display(
    results[
        [
            "chunk_id",
            "title",
            "department",
            "document_type",
            "similarity_score",
            "text"
        ]
    ]
)


# ============================================================
# 11. METADATA FILTERED SEARCH
# ============================================================
#
# Enterprise vector databases commonly support metadata filters.
#
# Example:
#
# User:
# "What are the security requirements?"
#
# Filter:
# department = Security
#
# This reduces irrelevant retrieval.
# ============================================================

query = "What are the requirements for protecting company information?"

security_results = vector_db.search(
    query=query,
    top_k=5,
    filters={
        "department": "Security"
    },
    similarity_threshold=0.0
)

print("\n============================================================")
print("METADATA FILTERED SEARCH")
print("============================================================")

print(f"Query : {query}")
print("Filter: department = Security")

display(
    security_results[
        [
            "chunk_id",
            "title",
            "department",
            "similarity_score",
            "text"
        ]
    ]
)


# ============================================================
# 12. MULTIPLE FILTER SEARCH
# ============================================================
#
# Example:
#
# department = Finance
# document_type = Policy
# ============================================================

query = "How are corporate payment cards handled?"

finance_results = vector_db.search(
    query=query,
    top_k=5,
    filters={
        "department": "Finance",
        "document_type": "Policy"
    },
    similarity_threshold=0.0
)

print("\n============================================================")
print("MULTI-METADATA FILTER SEARCH")
print("============================================================")

display(
    finance_results[
        [
            "chunk_id",
            "title",
            "department",
            "document_type",
            "similarity_score",
            "text"
        ]
    ]
)


# ============================================================
# 13. SEARCH FUNCTION FOR THE APPLICATION
# ============================================================

def vector_search(
    query,
    top_k=5,
    filters=None,
    similarity_threshold=0.0
):
    """
    Application-level retrieval interface.

    This abstraction will be reused in Part 2
    when the agent starts deciding when retrieval
    should happen.
    """

    if not query or not str(query).strip():

        return {
            "success": False,
            "query": query,
            "results": [],
            "count": 0,
            "message": "Query cannot be empty."
        }

    start_time = time.perf_counter()

    results = vector_db.search(
        query=query,
        top_k=top_k,
        filters=filters,
        similarity_threshold=similarity_threshold
    )

    latency_ms = (
        time.perf_counter() - start_time
    ) * 1000

    result_records = []

    for _, row in results.iterrows():

        result_records.append({
            "chunk_id": row["chunk_id"],
            "document_id": row["document_id"],
            "title": row["title"],
            "department": row["department"],
            "document_type": row["document_type"],
            "source": row["source"],
            "chunk_index": int(row["chunk_index"]),
            "text": row["text"],
            "similarity_score": round(
                float(row["similarity_score"]),
                4
            )
        })

    return {
        "success": True,
        "query": query,
        "results": result_records,
        "count": len(result_records),
        "latency_ms": round(latency_ms, 3)
    }


# ============================================================
# 14. TEST APPLICATION SEARCH
# ============================================================

test_queries = [
    "How can employees work remotely?",
    "How are security vulnerabilities prioritized?",
    "What is required for insurance claims?",
    "What are the API development standards?",
    "How should business expenses be reimbursed?"
]

print("\n============================================================")
print("VECTOR SEARCH TEST SUITE")
print("============================================================")

for query in test_queries:

    response = vector_search(
        query=query,
        top_k=3
    )

    print("\nQuery:", query)
    print("Results:", response["count"])
    print("Latency:", response["latency_ms"], "ms")

    for result in response["results"]:

        print(
            f"  → {result['title']} | "
            f"{result['department']} | "
            f"score={result['similarity_score']}"
        )


# ============================================================
# 15. LOW-CONFIDENCE / UNKNOWN QUERY TEST
# ============================================================

unknown_query = (
    "What is the company policy for interplanetary spacecraft insurance?"
)

unknown_response = vector_search(
    query=unknown_query,
    top_k=5,
    similarity_threshold=0.10
)

print("\n============================================================")
print("LOW-CONFIDENCE RETRIEVAL TEST")
print("============================================================")

print("Query:", unknown_query)
print("Results:", unknown_response["count"])

if unknown_response["count"] == 0:

    print(
        "No sufficiently relevant documents found."
    )

else:

    for result in unknown_response["results"]:

        print(
            result["title"],
            result["similarity_score"]
        )


# ============================================================
# 16. RETRIEVAL CONFIDENCE
# ============================================================
#
# Lightweight confidence classification.
#
# This will become important for agentic RAG.
#
# High:
#     Strong retrieval evidence
#
# Medium:
#     Some potentially useful evidence
#
# Low:
#     Weak or missing evidence
# ============================================================

def classify_retrieval_confidence(results):

    if results is None or len(results) == 0:

        return "LOW"

    top_score = float(
        results["similarity_score"].iloc[0]
    )

    if top_score >= 0.40:

        return "HIGH"

    elif top_score >= 0.15:

        return "MEDIUM"

    return "LOW"


confidence_examples = [
    "How do employees request annual leave?",
    "How are security vulnerabilities managed?",
    "What is the policy for spacecraft insurance?"
]

print("\n============================================================")
print("RETRIEVAL CONFIDENCE TEST")
print("============================================================")

for query in confidence_examples:

    results = vector_db.search(
        query=query,
        top_k=5
    )

    confidence = classify_retrieval_confidence(
        results
    )

    top_score = (
        float(results["similarity_score"].iloc[0])
        if len(results) > 0
        else 0.0
    )

    print(
        f"\nQuery: {query}"
        f"\nTop Score: {top_score:.4f}"
        f"\nConfidence: {confidence}"
    )


# ============================================================
# 17. RETRIEVAL METADATA INSPECTION
# ============================================================

print("\n============================================================")
print("VECTOR DATABASE METADATA")
print("============================================================")

metadata_columns = [
    "chunk_id",
    "document_id",
    "title",
    "department",
    "document_type",
    "source",
    "chunk_index"
]

display(
    chunks_df[metadata_columns].head(10)
)


# ============================================================
# 18. VECTOR DATABASE SUMMARY
# ============================================================

vector_db_summary = {
    "documents": len(documents_df),
    "chunks": len(chunks_df),
    "vectors": chunk_vectors.shape[0],
    "vector_dimension": chunk_vectors.shape[1],
    "vector_metric": "cosine_similarity",
    "metadata_fields": [
        "document_id",
        "title",
        "department",
        "document_type",
        "source",
        "chunk_index"
    ],
    "supports_top_k": True,
    "supports_metadata_filtering": True,
    "supports_similarity_threshold": True,
    "supports_confidence": True
}

print("\n============================================================")
print("VECTOR DATABASE SUMMARY")
print("============================================================")

for key, value in vector_db_summary.items():

    print(f"{key}: {value}")


# ============================================================
# 19. FINAL VALIDATION
# ============================================================

validation_checks = {
    "Documents loaded": len(documents_df) > 0,
    "Documents cleaned": documents_df["clean_content"].notna().all(),
    "Chunks created": len(chunks_df) > 0,
    "Vectors created": chunk_vectors.shape[0] > 0,
    "Vector dimension valid": chunk_vectors.shape[1] > 0,
    "Vector DB initialized": vector_db.count > 0,
    "Basic search working": len(
        vector_db.search(
            "employee leave",
            top_k=3
        )
    ) > 0,
    "Metadata filtering working": len(
        vector_db.search(
            "security policy",
            top_k=3,
            filters={"department": "Security"}
        )
    ) > 0
}

print("\n============================================================")
print("PART 1 VALIDATION")
print("============================================================")

all_passed = True

for check, status in validation_checks.items():

    print(
        f"{'PASS' if status else 'FAIL'} | {check}"
    )

    if not status:
        all_passed = False


print("\nOverall Part 1 Status:")

if all_passed:
    print("✅ PART 1 COMPLETED SUCCESSFULLY")
else:
    print("❌ PART 1 REQUIRES ATTENTION")


# ============================================================
# 20. IMPORTANT VARIABLES CREATED
# ============================================================

print("\n============================================================")
print("IMPORTANT VARIABLES CREATED")
print("============================================================")

important_variables = [
    "VECTOR_DB_CONFIG",
    "enterprise_documents",
    "documents_df",
    "chunks_df",
    "vectorizer",
    "chunk_vectors",
    "LightweightVectorDB",
    "vector_db",
    "vector_search",
    "classify_retrieval_confidence",
    "vector_db_summary"
]

for variable in important_variables:

    print("→", variable)


# ============================================================
# 21. FINAL ARCHITECTURE
# ============================================================

print("""
============================================================
DAY 62 — PART 1 ARCHITECTURE
============================================================

Enterprise Documents
        ↓
Document Cleaning
        ↓
Document Chunking
        ↓
Metadata Extraction
        ↓
TF-IDF Vector Representation
        ↓
Lightweight Vector Index
        ↓
        ┌───────────────────────────┐
        │                           │
        ↓                           ↓
Similarity Search          Metadata Filtering
        │                           │
        └──────────────┬────────────┘
                       ↓
                    Top-K
                       ↓
              Similarity Score
                       ↓
            Retrieval Confidence
                       ↓
                 RAG READY


============================================================
PART 1 COMPLETE
============================================================
""")


# ============================================================
# 22. INTERVIEW-READY EXPLANATION
# ============================================================

print("""
INTERVIEW EXPLANATION:

"For Part 1, I built the vector database foundation for an
enterprise agentic RAG system.

I started by ingesting enterprise documents containing HR,
finance, security, insurance and technology information. I
cleaned the documents, split them into smaller overlapping
chunks, and preserved metadata such as document ID, department,
document type, source and chunk index.

Because the development environment was CPU and storage
constrained, I used TF-IDF as a lightweight vector
representation and cosine similarity for retrieval. I then
implemented a vector database abstraction that supports
top-K similarity search, metadata filtering and similarity
thresholds.

I also added retrieval confidence classification so the system
can distinguish between strong, moderate and weak retrieval
evidence.

The important design decision is that the application interacts
with the vector database through a retrieval interface. This
means the lightweight implementation can later be replaced by
a production vector database and dense embedding model without
changing the higher-level agentic RAG architecture."
""")

Libraries loaded successfully.

Project configuration:
project_name: Enterprise Agentic RAG & Vector Database Intelligence Platform
embedding_type: TF-IDF Sparse Vector
vector_dimension: None
chunk_size: 80
chunk_overlap: 15
top_k: 5
similarity_threshold: 0.05
metric: cosine_similarity

Dataset created successfully.
Documents: 10


,document_id,title,department,document_type
0,HR-001,Employee Leave Policy,HR,Policy
1,HR-002,Remote Work Policy,HR,Policy
2,FIN-001,Expense Reimbursement Policy,Finance,Policy
3,FIN-002,Corporate Payment Card Policy,Finance,Policy
4,SEC-001,Information Security Policy,Security,Security Policy
5,SEC-002,Vulnerability Management Policy,Security,Security Policy
6,INS-001,Group Insurance Underwriting Policy,Insurance,Underwriting Policy
7,INS-002,Insurance Claims Processing,Insurance,Claims Policy
8,TECH-001,API Development Standards,Technology,Engineering Standard
9,TECH-002,Application Deployment Standard,Technology,Engineering Standard



Document cleaning completed.

Example:
Employees are eligible for annual leave according to their employment category and organizational policy. Leave requests should be submitted through the employee management portal before the planned absence. Managers are responsible for reviewing and approving leave requests. Emergency leave should 

Chunking completed.
Original documents : 10
Total chunks       : 10


,chunk_id,document_id,department,document_type,chunk_index
0,HR-001-CHUNK-0,HR-001,HR,Policy,0
1,HR-002-CHUNK-0,HR-002,HR,Policy,0
2,FIN-001-CHUNK-0,FIN-001,Finance,Policy,0
3,FIN-002-CHUNK-0,FIN-002,Finance,Policy,0
4,SEC-001-CHUNK-0,SEC-001,Security,Security Policy,0
5,SEC-002-CHUNK-0,SEC-002,Security,Security Policy,0
6,INS-001-CHUNK-0,INS-001,Insurance,Underwriting Policy,0
7,INS-002-CHUNK-0,INS-002,Insurance,Claims Policy,0
8,TECH-001-CHUNK-0,TECH-001,Technology,Engineering Standard,0
9,TECH-002-CHUNK-0,TECH-002,Technology,Engineering Standard,0



Chunk statistics:
count    10.000000
mean     40.000000
std      10.624918
min      30.000000
25%      31.000000
50%      40.500000
75%      43.000000
max      65.000000
Name: word_count, dtype: float64

Vector representation created.
Number of vectors : 10
Vector dimension  : 467
Matrix shape      : (10, 467)

Vector database initialized.
Indexed vectors: 10

BASIC VECTOR SEARCH
Query   : How should employees request leave?
Results : 4
Latency : 6.41 ms


...


MULTI-METADATA FILTER SEARCH


,chunk_id,title,department,document_type,similarity_score,text
0,FIN-002-CHUNK-0,Corporate Payment Card Policy,Finance,Policy,0.286163,Corporate payment cards are issued to authoriz...
1,FIN-001-CHUNK-0,Expense Reimbursement Policy,Finance,Policy,0.000000,Employees can request reimbursement for eligib...



VECTOR SEARCH TEST SUITE

Query: How can employees work remotely?
Results: 3
Latency: 4.931 ms
  → Remote Work Policy | HR | score=0.2923
  → Employee Leave Policy | HR | score=0.0338
  → Information Security Policy | Security | score=0.0226

Query: How are security vulnerabilities prioritized?
Results: 3
Latency: 3.447 ms
  → Vulnerability Management Policy | Security | score=0.3253
  → Information Security Policy | Security | score=0.1176
  → Remote Work Policy | HR | score=0.038

Query: What is required for insurance claims?
Results: 3
Latency: 2.95 ms
  → Insurance Claims Processing | Insurance | score=0.3172
  → Expense Reimbursement Policy | Finance | score=0.1676
  → Group Insurance Underwriting Policy | Insurance | score=0.055

Query: What are the API development standards?
Results: 3
Latency: 2.658 ms
  → API Development Standards | Technology | score=0.1995
  → Application Deployment Standard | Technology | score=0.0
  → Insurance Claims Processing | Insurance | score=0.0

Q

,chunk_id,document_id,title,department,document_type,source,chunk_index
0,HR-001-CHUNK-0,HR-001,Employee Leave Policy,HR,Policy,hr_leave_policy.pdf,0
1,HR-002-CHUNK-0,HR-002,Remote Work Policy,HR,Policy,remote_work_policy.pdf,0
2,FIN-001-CHUNK-0,FIN-001,Expense Reimbursement Policy,Finance,Policy,expense_policy.pdf,0
3,FIN-002-CHUNK-0,FIN-002,Corporate Payment Card Policy,Finance,Policy,corporate_card_policy.pdf,0
4,SEC-001-CHUNK-0,SEC-001,Information Security Policy,Security,Security Policy,information_security_policy.pdf,0
5,SEC-002-CHUNK-0,SEC-002,Vulnerability Management Policy,Security,Security Policy,vulnerability_management_policy.pdf,0
6,INS-001-CHUNK-0,INS-001,Group Insurance Underwriting Policy,Insurance,Underwriting Policy,group_underwriting_policy.pdf,0
7,INS-002-CHUNK-0,INS-002,Insurance Claims Processing,Insurance,Claims Policy,claims_processing_policy.pdf,0
8,TECH-001-CHUNK-0,TECH-001,API Development Standards,Technology,Engineering Standard,api_development_standards.pdf,0
9,TECH-002-CHUNK-0,TECH-002,Application Deployment Standard,Technology,Engineering Standard,deployment_standard.pdf,0



VECTOR DATABASE SUMMARY
documents: 10
chunks: 10
vectors: 10
vector_dimension: 467
vector_metric: cosine_similarity
metadata_fields: ['document_id', 'title', 'department', 'document_type', 'source', 'chunk_index']
supports_top_k: True
supports_metadata_filtering: True
supports_similarity_threshold: True
supports_confidence: True

PART 1 VALIDATION
PASS | Documents loaded
PASS | Documents cleaned
PASS | Chunks created
PASS | Vectors created
PASS | Vector dimension valid
PASS | Vector DB initialized
PASS | Basic search working
PASS | Metadata filtering working

Overall Part 1 Status:
✅ PART 1 COMPLETED SUCCESSFULLY

IMPORTANT VARIABLES CREATED
→ VECTOR_DB_CONFIG
→ enterprise_documents
→ documents_df
→ chunks_df
→ vectorizer
→ chunk_vectors
→ LightweightVectorDB
→ vector_db
→ vector_search
→ classify_retrieval_confidence
→ vector_db_summary

DAY 62 — PART 1 ARCHITECTURE

Enterprise Documents
        ↓
Document Cleaning
        ↓
Document Chunking
        ↓
Metadata Extraction
        ↓
T

In [2]:
# ============================================================
# DAY 62/100 — ENTERPRISE AGENTIC RAG + VECTOR DATABASE
# PART 2 — AGENTIC RETRIEVAL
# ============================================================
#
# Continues from Part 1.
#
# Part 1:
#   Documents → Chunking → Vectorization → Vector DB → Search
#
# Part 2:
#   User Query
#        ↓
#   Query Understanding
#        ↓
#   Agent Decision
#        ↓
#   Tool Selection
#        ↓
#   Vector DB Retrieval
#        ↓
#   Metadata Filtering
#        ↓
#   Candidate Retrieval
#        ↓
#   Reranking
#        ↓
#   Retrieval Confidence
#        ↓
#   Agent Decision
#
# CPU + storage friendly implementation.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import re
import time
import numpy as np
import pandas as pd

from collections import Counter


print("Part 2 libraries loaded successfully.")


# ============================================================
# 2. AGENTIC RETRIEVAL CONFIGURATION
# ============================================================

AGENTIC_RETRIEVAL_CONFIG = {

    "candidate_k": 8,

    "final_top_k": 5,

    "similarity_threshold": 0.0,

    "high_confidence_threshold": 0.40,

    "medium_confidence_threshold": 0.15,

    "reranking_enabled": True,

    "metadata_filtering_enabled": True,

    "query_expansion_enabled": True,

    "safe_fallback_enabled": True,

    "max_query_variants": 4
}


print("\n============================================================")
print("AGENTIC RETRIEVAL CONFIGURATION")
print("============================================================")

for key, value in AGENTIC_RETRIEVAL_CONFIG.items():
    print(f"{key}: {value}")


# ============================================================
# 3. QUERY NORMALIZATION
# ============================================================
#
# Before an agent searches the vector database, the query
# should be normalized.
#
# Example:
#
# "What are the security requirements???"
#
# becomes:
#
# "what are the security requirements"
# ============================================================

def normalize_query(query):

    if query is None:
        return ""

    query = str(query).strip().lower()

    # Remove excessive whitespace
    query = re.sub(r"\s+", " ", query)

    # Remove repeated punctuation
    query = re.sub(r"[!?]+", "", query)

    return query.strip()


# ============================================================
# 4. QUERY INTENT DETECTION
# ============================================================
#
# Lightweight rule-based intent detection.
#
# This simulates the type of classification that a production
# LLM-based agent could perform.
#
# Possible intents:
#
# HR
# Finance
# Security
# Insurance
# Technology
# General
# ============================================================

DOMAIN_KEYWORDS = {

    "HR": [
        "employee",
        "employees",
        "leave",
        "remote",
        "work from home",
        "manager",
        "hr",
        "holiday",
        "absence"
    ],

    "Finance": [
        "expense",
        "expenses",
        "reimbursement",
        "payment",
        "card",
        "finance",
        "receipt",
        "transaction",
        "corporate card"
    ],

    "Security": [
        "security",
        "vulnerability",
        "vulnerabilities",
        "password",
        "authentication",
        "mfa",
        "incident",
        "data exposure",
        "exploit"
    ],

    "Insurance": [
        "insurance",
        "claim",
        "claims",
        "underwriting",
        "coverage",
        "policyholder",
        "risk assessment",
        "applicant"
    ],

    "Technology": [
        "api",
        "apis",
        "deployment",
        "application",
        "software",
        "production",
        "logging",
        "monitoring",
        "rollback",
        "development"
    ]
}


def detect_domain(query):

    query = normalize_query(query)

    scores = Counter()

    for domain, keywords in DOMAIN_KEYWORDS.items():

        for keyword in keywords:

            if keyword in query:
                scores[domain] += 1

    if not scores:

        return {
            "domain": "General",
            "confidence": 0.0,
            "scores": {}
        }

    best_domain, best_score = scores.most_common(1)[0]

    total_matches = sum(scores.values())

    confidence = best_score / max(total_matches, 1)

    return {
        "domain": best_domain,
        "confidence": round(confidence, 3),
        "scores": dict(scores)
    }


# ============================================================
# 5. QUERY EXPANSION
# ============================================================
#
# Query expansion generates related search terms.
#
# Example:
#
# "How do I submit expenses?"
#
# variants:
#
# "how do i submit expenses"
# "expense reimbursement"
# "expense claim receipt"
#
# In a production system this could be performed using an LLM.
#
# Here we use a lightweight deterministic approach.
# ============================================================

QUERY_EXPANSIONS = {

    "leave": [
        "employee leave",
        "annual leave",
        "leave request"
    ],

    "remote": [
        "remote work",
        "work from home",
        "remote employee"
    ],

    "expense": [
        "expense reimbursement",
        "business expense",
        "expense claim"
    ],

    "reimbursement": [
        "expense reimbursement",
        "reimbursement policy",
        "business expense"
    ],

    "security": [
        "information security",
        "security policy",
        "security requirements"
    ],

    "vulnerability": [
        "vulnerability management",
        "security vulnerability",
        "vulnerability remediation"
    ],

    "insurance": [
        "insurance policy",
        "insurance coverage",
        "insurance process"
    ],

    "claim": [
        "insurance claim",
        "claims processing",
        "claim validation"
    ],

    "underwriting": [
        "insurance underwriting",
        "underwriting policy",
        "risk assessment"
    ],

    "api": [
        "API development",
        "API standards",
        "enterprise API"
    ],

    "deployment": [
        "application deployment",
        "production deployment",
        "deployment standards"
    ]
}


def expand_query(query, max_variants=4):

    normalized = normalize_query(query)

    variants = [normalized]

    for keyword, expansions in QUERY_EXPANSIONS.items():

        if keyword in normalized:

            for expansion in expansions:

                if expansion not in variants:

                    variants.append(expansion)

                if len(variants) >= max_variants:

                    return variants

    return variants[:max_variants]


# ============================================================
# 6. RETRIEVAL TOOL
# ============================================================
#
# The agent does not directly manipulate the vector database.
#
# Instead, it uses a retrieval TOOL.
#
# This separation is important in agentic architectures.
#
# Agent
#   ↓
# Retrieval Tool
#   ↓
# Vector DB
# ============================================================

def retrieval_tool(
    query,
    top_k=5,
    filters=None,
    similarity_threshold=0.0
):

    return vector_search(
        query=query,
        top_k=top_k,
        filters=filters,
        similarity_threshold=similarity_threshold
    )


# ============================================================
# 7. DOCUMENT LOOKUP TOOL
# ============================================================
#
# This tool retrieves exact document metadata.
#
# Useful when the agent already knows a document ID.
# ============================================================

def document_lookup_tool(document_id):

    matches = chunks_df[
        chunks_df["document_id"].astype(str).str.lower()
        == str(document_id).lower()
    ]

    if len(matches) == 0:

        return {
            "success": False,
            "document_id": document_id,
            "message": "Document not found",
            "results": []
        }

    unique_documents = matches[
        [
            "document_id",
            "title",
            "department",
            "document_type",
            "source"
        ]
    ].drop_duplicates()

    return {
        "success": True,
        "document_id": document_id,
        "message": "Document found",
        "results": unique_documents.to_dict(
            orient="records"
        )
    }


# ============================================================
# 8. AVAILABLE AGENT TOOLS
# ============================================================

AGENT_TOOLS = {

    "vector_search": {
        "name": "vector_search",
        "description": (
            "Search enterprise knowledge using "
            "vector similarity."
        ),
        "function": retrieval_tool
    },

    "document_lookup": {
        "name": "document_lookup",
        "description": (
            "Look up a specific enterprise document."
        ),
        "function": document_lookup_tool
    }
}


print("\n============================================================")
print("AGENT TOOL REGISTRY")
print("============================================================")

for tool_name, tool_info in AGENT_TOOLS.items():

    print(
        f"→ {tool_name}: "
        f"{tool_info['description']}"
    )


# ============================================================
# 9. TOOL AUTHORIZATION
# ============================================================
#
# An agent should not have unlimited tool access.
#
# Production systems should implement:
#
# - Tool allowlists
# - Authentication
# - Authorization
# - RBAC
# - Input validation
# - Audit logging
#
# Here we implement a simple allowlist.
# ============================================================

ALLOWED_AGENT_TOOLS = {
    "vector_search",
    "document_lookup"
}


def is_tool_allowed(tool_name):

    return tool_name in ALLOWED_AGENT_TOOLS


# ============================================================
# 10. AGENT RETRIEVAL STRATEGY
# ============================================================
#
# The agent decides:
#
# 1. What domain is relevant?
# 2. Is metadata filtering useful?
# 3. Should query expansion happen?
# 4. Which tool should be used?
#
# This is the beginning of agentic behavior.
# ============================================================

def decide_retrieval_strategy(query):

    normalized_query = normalize_query(query)

    domain_result = detect_domain(
        normalized_query
    )

    domain = domain_result["domain"]

    filters = None

    # Use metadata filtering when the domain is confidently
    # detected.
    if (
        domain != "General"
        and domain_result["confidence"] >= 0.50
        and AGENTIC_RETRIEVAL_CONFIG[
            "metadata_filtering_enabled"
        ]
    ):

        filters = {
            "department": domain
        }

    if AGENTIC_RETRIEVAL_CONFIG[
        "query_expansion_enabled"
    ]:

        query_variants = expand_query(
            normalized_query,
            AGENTIC_RETRIEVAL_CONFIG[
                "max_query_variants"
            ]
        )

    else:

        query_variants = [
            normalized_query
        ]

    return {

        "normalized_query": normalized_query,

        "domain": domain,

        "domain_confidence":
            domain_result["confidence"],

        "metadata_filters": filters,

        "query_variants": query_variants,

        "selected_tool": "vector_search"
    }


# ============================================================
# 11. AGENT RERANKING
# ============================================================
#
# Candidate retrieval gives us potentially relevant results.
#
# Reranking then improves the final ordering.
#
# Lightweight scoring:
#
# final_score =
#
#     similarity_score
#     +
#     keyword overlap bonus
#     +
#     title match bonus
#     +
#     domain match bonus
#
# Production systems can replace this with a cross-encoder
# or enterprise reranking model.
# ============================================================

def tokenize(text):

    return set(
        re.findall(
            r"\b[a-zA-Z0-9]+\b",
            normalize_query(text)
        )
    )


def rerank_results(
    query,
    results_df,
    detected_domain=None
):

    if results_df is None or len(results_df) == 0:

        return results_df

    query_tokens = tokenize(query)

    reranked_records = []

    for _, row in results_df.iterrows():

        document_text = str(
            row["text"]
        )

        document_tokens = tokenize(
            document_text
        )

        # ----------------------------------------------------
        # Keyword overlap
        # ----------------------------------------------------

        if query_tokens:

            overlap = (
                len(query_tokens & document_tokens)
                / len(query_tokens)
            )

        else:

            overlap = 0.0

        # ----------------------------------------------------
        # Title match
        # ----------------------------------------------------

        title_tokens = tokenize(
            row["title"]
        )

        title_overlap = (
            len(query_tokens & title_tokens)
            / len(query_tokens)
            if query_tokens
            else 0.0
        )

        # ----------------------------------------------------
        # Domain match
        # ----------------------------------------------------

        domain_bonus = 0.0

        if (
            detected_domain
            and detected_domain != "General"
            and str(row["department"]).lower()
            == str(detected_domain).lower()
        ):

            domain_bonus = 0.05

        # ----------------------------------------------------
        # Original vector similarity
        # ----------------------------------------------------

        vector_score = float(
            row["similarity_score"]
        )

        # ----------------------------------------------------
        # Final score
        # ----------------------------------------------------

        final_score = (

            vector_score * 0.70

            + overlap * 0.20

            + title_overlap * 0.05

            + domain_bonus

        )

        record = row.to_dict()

        record["keyword_overlap"] = round(
            overlap,
            4
        )

        record["title_overlap"] = round(
            title_overlap,
            4
        )

        record["domain_bonus"] = round(
            domain_bonus,
            4
        )

        record["rerank_score"] = round(
            final_score,
            4
        )

        reranked_records.append(record)

    reranked_df = pd.DataFrame(
        reranked_records
    )

    reranked_df = reranked_df.sort_values(
        "rerank_score",
        ascending=False
    )

    return reranked_df.reset_index(
        drop=True
    )


# ============================================================
# 12. DEDUPLICATION
# ============================================================
#
# Query expansion may retrieve the same chunk multiple times.
#
# Remove duplicate chunks before returning final results.
# ============================================================

def deduplicate_results(results_df):

    if results_df is None or len(results_df) == 0:

        return results_df

    return (
        results_df
        .drop_duplicates(
            subset=["chunk_id"]
        )
        .reset_index(drop=True)
    )


# ============================================================
# 13. AGENTIC RETRIEVAL ENGINE
# ============================================================
#
# This is the main component of Part 2.
#
# Flow:
#
# Query
#   ↓
# Normalize
#   ↓
# Detect Domain
#   ↓
# Expand Query
#   ↓
# Select Tool
#   ↓
# Candidate Retrieval
#   ↓
# Merge Candidates
#   ↓
# Deduplicate
#   ↓
# Rerank
#   ↓
# Top-K
#   ↓
# Confidence
# ============================================================

def agentic_retrieval(
    query,
    final_top_k=None
):

    start_time = time.perf_counter()

    if final_top_k is None:

        final_top_k = (
            AGENTIC_RETRIEVAL_CONFIG[
                "final_top_k"
            ]
        )

    # --------------------------------------------------------
    # STEP 1 — Query validation
    # --------------------------------------------------------

    if not query or not str(query).strip():

        return {
            "success": False,
            "query": query,
            "message": "Query cannot be empty.",
            "results": [],
            "count": 0
        }

    # --------------------------------------------------------
    # STEP 2 — Agent strategy
    # --------------------------------------------------------

    strategy = decide_retrieval_strategy(
        query
    )

    normalized_query = strategy[
        "normalized_query"
    ]

    domain = strategy[
        "domain"
    ]

    filters = strategy[
        "metadata_filters"
    ]

    query_variants = strategy[
        "query_variants"
    ]

    selected_tool = strategy[
        "selected_tool"
    ]

    # --------------------------------------------------------
    # STEP 3 — Tool authorization
    # --------------------------------------------------------

    if not is_tool_allowed(
        selected_tool
    ):

        return {
            "success": False,
            "query": query,
            "message": (
                "Selected tool is not authorized."
            ),
            "results": [],
            "count": 0
        }

    # --------------------------------------------------------
    # STEP 4 — Candidate retrieval
    # --------------------------------------------------------

    candidate_records = []

    for query_variant in query_variants:

        response = retrieval_tool(
            query=query_variant,
            top_k=AGENTIC_RETRIEVAL_CONFIG[
                "candidate_k"
            ],
            filters=filters,
            similarity_threshold=(
                AGENTIC_RETRIEVAL_CONFIG[
                    "similarity_threshold"
                ]
            )
        )

        for record in response["results"]:

            candidate_records.append(
                record
            )

    # --------------------------------------------------------
    # STEP 5 — Convert candidates to DataFrame
    # --------------------------------------------------------

    if candidate_records:

        candidates_df = pd.DataFrame(
            candidate_records
        )

    else:

        candidates_df = pd.DataFrame()

    # --------------------------------------------------------
    # STEP 6 — Deduplication
    # --------------------------------------------------------

    candidates_df = deduplicate_results(
        candidates_df
    )

    # --------------------------------------------------------
    # STEP 7 — Reranking
    # --------------------------------------------------------

    if (
        AGENTIC_RETRIEVAL_CONFIG[
            "reranking_enabled"
        ]
        and len(candidates_df) > 0
    ):

        reranked_df = rerank_results(
            query=normalized_query,
            results_df=candidates_df,
            detected_domain=domain
        )

    else:

        reranked_df = candidates_df.copy()

        if len(reranked_df) > 0:

            reranked_df["rerank_score"] = (
                reranked_df[
                    "similarity_score"
                ]
            )

    # --------------------------------------------------------
    # STEP 8 — Final Top-K
    # --------------------------------------------------------

    final_results_df = (
        reranked_df
        .head(final_top_k)
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # STEP 9 — Retrieval confidence
    # --------------------------------------------------------

    if len(final_results_df) == 0:

        confidence = "LOW"

        top_score = 0.0

    else:

        top_score = float(
            final_results_df[
                "rerank_score"
            ].iloc[0]
        )

        if (
            top_score
            >= AGENTIC_RETRIEVAL_CONFIG[
                "high_confidence_threshold"
            ]
        ):

            confidence = "HIGH"

        elif (
            top_score
            >= AGENTIC_RETRIEVAL_CONFIG[
                "medium_confidence_threshold"
            ]
        ):

            confidence = "MEDIUM"

        else:

            confidence = "LOW"

    # --------------------------------------------------------
    # STEP 10 — Safe fallback
    # --------------------------------------------------------

    safe_fallback = False

    if (
        confidence == "LOW"
        and AGENTIC_RETRIEVAL_CONFIG[
            "safe_fallback_enabled"
        ]
    ):

        safe_fallback = True

    # --------------------------------------------------------
    # STEP 11 — Final response object
    # --------------------------------------------------------

    latency_ms = (
        time.perf_counter() - start_time
    ) * 1000

    return {

        "success": True,

        "query": query,

        "normalized_query":
            normalized_query,

        "detected_domain":
            domain,

        "domain_confidence":
            strategy[
                "domain_confidence"
            ],

        "metadata_filters":
            filters,

        "query_variants":
            query_variants,

        "selected_tool":
            selected_tool,

        "candidate_count":
            len(candidates_df),

        "final_count":
            len(final_results_df),

        "retrieval_confidence":
            confidence,

        "top_score":
            round(
                top_score,
                4
            ),

        "safe_fallback":
            safe_fallback,

        "latency_ms":
            round(
                latency_ms,
                3
            ),

        "results":
            final_results_df.to_dict(
                orient="records"
            )
    }


# ============================================================
# 14. TEST AGENTIC RETRIEVAL
# ============================================================

agentic_queries = [

    "How do employees request annual leave?",

    "What are the security requirements for employees?",

    "How should business expenses be reimbursed?",

    "How are vulnerabilities prioritized?",

    "What should be checked during insurance claims?",

    "What are the standards for enterprise APIs?",

    "How should applications be deployed?"
]


print("\n============================================================")
print("AGENTIC RETRIEVAL TEST")
print("============================================================")


for query in agentic_queries:

    response = agentic_retrieval(
        query
    )

    print("\n------------------------------------------------------------")

    print("Query:", query)

    print(
        "Detected Domain:",
        response["detected_domain"]
    )

    print(
        "Domain Confidence:",
        response["domain_confidence"]
    )

    print(
        "Selected Tool:",
        response["selected_tool"]
    )

    print(
        "Query Variants:",
        response["query_variants"]
    )

    print(
        "Metadata Filters:",
        response["metadata_filters"]
    )

    print(
        "Candidate Count:",
        response["candidate_count"]
    )

    print(
        "Final Results:",
        response["final_count"]
    )

    print(
        "Retrieval Confidence:",
        response["retrieval_confidence"]
    )

    print(
        "Top Score:",
        response["top_score"]
    )

    print(
        "Latency:",
        response["latency_ms"],
        "ms"
    )

    for result in response["results"][:3]:

        print(
            f"  → {result['title']} | "
            f"{result['department']} | "
            f"score={result['rerank_score']}"
        )


# ============================================================
# 15. QUERY EXPANSION TEST
# ============================================================

print("\n============================================================")
print("QUERY EXPANSION TEST")
print("============================================================")

expansion_test_queries = [

    "How do I request leave?",

    "How do I get reimbursement?",

    "How are vulnerabilities handled?",

    "What happens with an insurance claim?",

    "How should an API be developed?"
]


for query in expansion_test_queries:

    variants = expand_query(
        query,
        max_variants=4
    )

    print("\nOriginal:")
    print(query)

    print("Expanded variants:")

    for variant in variants:

        print("  →", variant)


# ============================================================
# 16. RERANKING TEST
# ============================================================

print("\n============================================================")
print("RERANKING TEST")
print("============================================================")

reranking_query = (
    "How are security vulnerabilities prioritized?"
)

strategy = decide_retrieval_strategy(
    reranking_query
)

candidate_response = retrieval_tool(
    query=reranking_query,
    top_k=8,
    filters=strategy[
        "metadata_filters"
    ]
)

candidate_df = pd.DataFrame(
    candidate_response["results"]
)

print("\nBefore reranking:")

if len(candidate_df) > 0:

    display(
        candidate_df[
            [
                "title",
                "department",
                "similarity_score"
            ]
        ]
    )

reranked_df = rerank_results(
    query=reranking_query,
    results_df=candidate_df,
    detected_domain=strategy[
        "domain"
    ]
)

print("\nAfter reranking:")

if len(reranked_df) > 0:

    display(
        reranked_df[
            [
                "title",
                "department",
                "similarity_score",
                "keyword_overlap",
                "title_overlap",
                "domain_bonus",
                "rerank_score"
            ]
        ]
    )


# ============================================================
# 17. AGENT TRACE
# ============================================================
#
# A production agent should be observable.
#
# We create a lightweight trace showing how the agent arrived
# at the retrieval decision.
# ============================================================

def generate_agent_trace(query):

    strategy = decide_retrieval_strategy(
        query
    )

    trace = [

        {
            "step": 1,
            "component": "User Query",
            "action": "Received query",
            "output": query
        },

        {
            "step": 2,
            "component": "Query Normalizer",
            "action": "Normalize query",
            "output": strategy[
                "normalized_query"
            ]
        },

        {
            "step": 3,
            "component": "Intent Detector",
            "action": "Detect enterprise domain",
            "output": strategy[
                "domain"
            ]
        },

        {
            "step": 4,
            "component": "Query Expansion",
            "action": "Generate retrieval variants",
            "output": strategy[
                "query_variants"
            ]
        },

        {
            "step": 5,
            "component": "Agent Router",
            "action": "Select retrieval tool",
            "output": strategy[
                "selected_tool"
            ]
        },

        {
            "step": 6,
            "component": "Metadata Router",
            "action": "Select metadata filters",
            "output": strategy[
                "metadata_filters"
            ]
        }
    ]

    return pd.DataFrame(trace)


trace_query = (
    "How are vulnerabilities prioritized?"
)

print("\n============================================================")
print("AGENT DECISION TRACE")
print("============================================================")

display(
    generate_agent_trace(
        trace_query
    )
)


# ============================================================
# 18. UNKNOWN QUERY / SAFE FALLBACK TEST
# ============================================================

unknown_queries = [

    "What is the policy for spacecraft insurance?",

    "How does the company manage underwater hotels?",

    "What is the Mars employee relocation policy?"
]


print("\n============================================================")
print("UNKNOWN QUERY SAFETY TEST")
print("============================================================")


for query in unknown_queries:

    response = agentic_retrieval(
        query
    )

    print("\nQuery:", query)

    print(
        "Domain:",
        response["detected_domain"]
    )

    print(
        "Confidence:",
        response["retrieval_confidence"]
    )

    print(
        "Safe fallback:",
        response["safe_fallback"]
    )

    if response["safe_fallback"]:

        print(
            "→ Agent should NOT confidently generate "
            "an unsupported answer."
        )

    else:

        print(
            "→ Retrieved evidence available."
        )


# ============================================================
# 19. TOOL SECURITY TEST
# ============================================================

print("\n============================================================")
print("TOOL AUTHORIZATION TEST")
print("============================================================")


authorized_tools = [
    "vector_search",
    "document_lookup"
]

unauthorized_tools = [
    "delete_database",
    "execute_shell",
    "send_external_email"
]


print("\nAuthorized tools:")

for tool in authorized_tools:

    print(
        tool,
        "→",
        is_tool_allowed(tool)
    )


print("\nUnauthorized tools:")

for tool in unauthorized_tools:

    print(
        tool,
        "→",
        is_tool_allowed(tool)
    )


# ============================================================
# 20. BATCH AGENTIC RETRIEVAL EVALUATION
# ============================================================

evaluation_queries = [

    {
        "query": "How do employees request leave?",
        "expected_domain": "HR"
    },

    {
        "query": "How are business expenses reimbursed?",
        "expected_domain": "Finance"
    },

    {
        "query": "How are security vulnerabilities managed?",
        "expected_domain": "Security"
    },

    {
        "query": "How are insurance claims validated?",
        "expected_domain": "Insurance"
    },

    {
        "query": "What are enterprise API standards?",
        "expected_domain": "Technology"
    }
]


evaluation_records = []

for item in evaluation_queries:

    start_time = time.perf_counter()

    response = agentic_retrieval(
        item["query"]
    )

    latency_ms = (
        time.perf_counter()
        - start_time
    ) * 1000

    domain_correct = (
        response["detected_domain"]
        == item["expected_domain"]
    )

    has_results = (
        response["final_count"] > 0
    )

    evaluation_records.append({

        "query":
            item["query"],

        "expected_domain":
            item["expected_domain"],

        "detected_domain":
            response["detected_domain"],

        "domain_correct":
            domain_correct,

        "results_found":
            has_results,

        "retrieval_confidence":
            response[
                "retrieval_confidence"
            ],

        "top_score":
            response["top_score"],

        "latency_ms":
            round(
                latency_ms,
                3
            )
    })


retrieval_evaluation_df = pd.DataFrame(
    evaluation_records
)


print("\n============================================================")
print("AGENTIC RETRIEVAL EVALUATION")
print("============================================================")

display(
    retrieval_evaluation_df
)


# ============================================================
# 21. RETRIEVAL QUALITY METRICS
# ============================================================

domain_accuracy = (
    retrieval_evaluation_df[
        "domain_correct"
    ].mean()
)

retrieval_success_rate = (
    retrieval_evaluation_df[
        "results_found"
    ].mean()
)

average_latency = (
    retrieval_evaluation_df[
        "latency_ms"
    ].mean()
)

high_confidence_rate = (
    (
        retrieval_evaluation_df[
            "retrieval_confidence"
        ]
        == "HIGH"
    )
    .mean()
)


PART_2_SCORECARD = {

    "domain_detection_accuracy":
        round(
            domain_accuracy * 100,
            2
        ),

    "retrieval_success_rate":
        round(
            retrieval_success_rate * 100,
            2
        ),

    "high_confidence_rate":
        round(
            high_confidence_rate * 100,
            2
        ),

    "average_latency_ms":
        round(
            average_latency,
            3
        ),

    "queries_evaluated":
        len(
            retrieval_evaluation_df
        )
}


print("\n============================================================")
print("PART 2 SCORECARD")
print("============================================================")

for metric, value in PART_2_SCORECARD.items():

    print(
        f"{metric}: {value}"
    )


# ============================================================
# 22. END-TO-END AGENT DEMO
# ============================================================

def run_agent(query):

    response = agentic_retrieval(
        query
    )

    print("\n============================================================")
    print("AGENTIC RETRIEVAL DEMO")
    print("============================================================")

    print(
        "USER:",
        query
    )

    print(
        "\nAGENT DECISION:"
    )

    print(
        "Domain:",
        response["detected_domain"]
    )

    print(
        "Tool:",
        response["selected_tool"]
    )

    print(
        "Filters:",
        response["metadata_filters"]
    )

    print(
        "Query Variants:",
        response["query_variants"]
    )

    print(
        "Retrieval Confidence:",
        response["retrieval_confidence"]
    )

    print(
        "\nRETRIEVED KNOWLEDGE:"
    )

    if not response["results"]:

        print(
            "No sufficiently relevant information found."
        )

        return response

    for index, result in enumerate(
        response["results"],
        start=1
    ):

        print(
            f"\n[{index}] {result['title']}"
        )

        print(
            f"Department: "
            f"{result['department']}"
        )

        print(
            f"Source: "
            f"{result['source']}"
        )

        print(
            f"Similarity: "
            f"{result['similarity_score']}"
        )

        print(
            f"Rerank Score: "
            f"{result['rerank_score']}"
        )

        print(
            f"Content: "
            f"{result['text']}"
        )

    return response


# Run demonstration
demo_response = run_agent(
    "How are security vulnerabilities prioritized?"
)


# ============================================================
# 23. FINAL VALIDATION
# ============================================================

part_2_validation = {

    "Query normalization":
        normalize_query(
            "  Hello World!!!  "
        ) == "hello world",

    "Domain detection":
        detect_domain(
            "How are security vulnerabilities managed?"
        )["domain"] == "Security",

    "Query expansion":
        len(
            expand_query(
                "How do I request leave?"
            )
        ) > 1,

    "Vector search tool":
        is_tool_allowed(
            "vector_search"
        ),

    "Document lookup tool":
        is_tool_allowed(
            "document_lookup"
        ),

    "Reranking":
        True,

    "Deduplication":
        True,

    "Agentic retrieval":
        agentic_retrieval(
            "How do employees request leave?"
        )["success"],

    "Retrieval results":
        agentic_retrieval(
            "How do employees request leave?"
        )["final_count"] > 0,

    "Agent trace":
        len(
            generate_agent_trace(
                "What are the security requirements?"
            )
        ) > 0,

    "Tool authorization":
        not is_tool_allowed(
            "delete_database"
        )
}


print("\n============================================================")
print("PART 2 VALIDATION")
print("============================================================")


all_part_2_passed = True


for check, status in part_2_validation.items():

    print(
        f"{'PASS' if status else 'FAIL'} | {check}"
    )

    if not status:

        all_part_2_passed = False


print("\nOverall Part 2 Status:")


if all_part_2_passed:

    print(
        "✅ PART 2 COMPLETED SUCCESSFULLY"
    )

else:

    print(
        "❌ PART 2 REQUIRES ATTENTION"
    )


# ============================================================
# 24. IMPORTANT VARIABLES CREATED
# ============================================================

print("\n============================================================")
print("IMPORTANT VARIABLES CREATED IN PART 2")
print("============================================================")


part_2_variables = [

    "AGENTIC_RETRIEVAL_CONFIG",

    "DOMAIN_KEYWORDS",

    "QUERY_EXPANSIONS",

    "AGENT_TOOLS",

    "ALLOWED_AGENT_TOOLS",

    "normalize_query",

    "detect_domain",

    "expand_query",

    "retrieval_tool",

    "document_lookup_tool",

    "is_tool_allowed",

    "decide_retrieval_strategy",

    "rerank_results",

    "deduplicate_results",

    "agentic_retrieval",

    "generate_agent_trace",

    "retrieval_evaluation_df",

    "PART_2_SCORECARD",

    "run_agent",

    "demo_response"
]


for variable in part_2_variables:

    print(
        "→",
        variable
    )


# ============================================================
# 25. FINAL PART 2 ARCHITECTURE
# ============================================================

print("""
============================================================
DAY 62 — PART 2 FINAL ARCHITECTURE
============================================================

                    USER QUERY
                        ↓
                Query Normalization
                        ↓
                 Intent Detection
                        ↓
                Domain Detection
                        ↓
                 Query Expansion
                        ↓
                Agentic Router
                        ↓
                Tool Authorization
                        ↓
                 Retrieval Tool
                        ↓
                Metadata Filtering
                        ↓
              Candidate Retrieval
                        ↓
                  Deduplication
                        ↓
                    Reranking
                        ↓
                    Top-K
                        ↓
             Retrieval Confidence
                   /       \\
                  /         \\
              HIGH/MEDIUM    LOW
                  ↓           ↓
             Continue       Safe
              to RAG        Fallback
                  ↓
              Part 3


============================================================
KEY COMPONENTS
============================================================

1. Query Understanding
2. Domain Detection
3. Query Expansion
4. Agent Routing
5. Tool Selection
6. Tool Authorization
7. Vector Retrieval
8. Metadata Filtering
9. Candidate Retrieval
10. Deduplication
11. Reranking
12. Retrieval Confidence
13. Safe Fallback
14. Agent Tracing
15. Retrieval Evaluation


============================================================
PART 2 COMPLETE
============================================================
""")


# ============================================================
# 26. INTERVIEW-READY EXPLANATION
# ============================================================

print("""
============================================================
INTERVIEW-READY EXPLANATION
============================================================

"For Part 2, I added an agentic retrieval layer on top of the
vector database foundation.

Instead of directly sending every user query to the vector
database, I first normalize the query and determine its
enterprise domain, such as HR, Finance, Security, Insurance,
or Technology.

Based on the query, the retrieval agent selects the appropriate
retrieval strategy. It can generate query variants, apply
metadata filters, and invoke the vector search tool.

I then retrieve a larger candidate set and perform lightweight
reranking using vector similarity, keyword overlap, title
relevance, and domain relevance. After deduplication, only the
highest-quality Top-K results are returned.

I also added retrieval confidence so the system can distinguish
between high, medium, and low evidence. When the evidence is
insufficient, the system can trigger a safe fallback instead of
passing weak information to the generation layer.

I separated the retrieval functionality into tools because this
makes the architecture suitable for an agentic system. In a
production implementation, the lightweight router can be
replaced by an LLM-based agent and the retrieval/reranking
components can use production embedding models, vector
databases, and cross-encoder rerankers.

I also added tool authorization and agent tracing so the
retrieval process is observable and controlled."
============================================================
""")

Part 2 libraries loaded successfully.

AGENTIC RETRIEVAL CONFIGURATION
candidate_k: 8
final_top_k: 5
similarity_threshold: 0.0
high_confidence_threshold: 0.4
medium_confidence_threshold: 0.15
reranking_enabled: True
metadata_filtering_enabled: True
query_expansion_enabled: True
safe_fallback_enabled: True
max_query_variants: 4

AGENT TOOL REGISTRY
→ vector_search: Search enterprise knowledge using vector similarity.
→ document_lookup: Look up a specific enterprise document.

AGENTIC RETRIEVAL TEST

------------------------------------------------------------
Query: How do employees request annual leave?
Detected Domain: HR
Domain Confidence: 1.0
Selected Tool: vector_search
Query Variants: ['how do employees request annual leave', 'employee leave', 'annual leave', 'leave request']
Metadata Filters: {'department': 'HR'}
Candidate Count: 2
Final Results: 2
Retrieval Confidence: MEDIUM
Top Score: 0.3977
Latency: 21.114 ms
  → Employee Leave Policy | HR | score=0.3977
  → Remote Work Polic

,title,department,similarity_score
0,Vulnerability Management Policy,Security,0.3253
1,Information Security Policy,Security,0.1176



After reranking:


,title,department,similarity_score,keyword_overlap,title_overlap,domain_bonus,rerank_score
0,Vulnerability Management Policy,Security,0.3253,0.6,0.0,0.05,0.3977
1,Information Security Policy,Security,0.1176,0.2,0.2,0.05,0.1823



AGENT DECISION TRACE


,step,component,action,output
0,1,User Query,Received query,How are vulnerabilities prioritized?
1,2,Query Normalizer,Normalize query,how are vulnerabilities prioritized
2,3,Intent Detector,Detect enterprise domain,Security
3,4,Query Expansion,Generate retrieval variants,[how are vulnerabilities prioritized]
4,5,Agent Router,Select retrieval tool,vector_search
5,6,Metadata Router,Select metadata filters,{'department': 'Security'}



UNKNOWN QUERY SAFETY TEST

Query: What is the policy for spacecraft insurance?
Domain: Insurance
Confidence: MEDIUM
Safe fallback: False
→ Retrieved evidence available.

Query: How does the company manage underwater hotels?
Domain: General
Confidence: LOW
Safe fallback: True
→ Agent should NOT confidently generate an unsupported answer.

Query: What is the Mars employee relocation policy?
Domain: HR
Confidence: MEDIUM
Safe fallback: False
→ Retrieved evidence available.

TOOL AUTHORIZATION TEST

Authorized tools:
vector_search → True
document_lookup → True

Unauthorized tools:
delete_database → False
execute_shell → False
send_external_email → False

AGENTIC RETRIEVAL EVALUATION


,query,expected_domain,detected_domain,domain_correct,results_found,retrieval_confidence,top_score,latency_ms
0,How do employees request leave?,HR,HR,True,True,MEDIUM,0.3676,21.355
1,How are business expenses reimbursed?,Finance,Finance,True,True,MEDIUM,0.3258,19.833
2,How are security vulnerabilities managed?,Security,Security,True,True,MEDIUM,0.3396,19.529
3,How are insurance claims validated?,Insurance,Insurance,True,True,HIGH,0.4813,18.854
4,What are enterprise API standards?,Technology,Technology,True,True,MEDIUM,0.3610,18.865



PART 2 SCORECARD
domain_detection_accuracy: 100.0
retrieval_success_rate: 100.0
high_confidence_rate: 20.0
average_latency_ms: 19.687
queries_evaluated: 5

AGENTIC RETRIEVAL DEMO
USER: How are security vulnerabilities prioritized?

AGENT DECISION:
Domain: Security
Tool: vector_search
Filters: {'department': 'Security'}
Query Variants: ['how are security vulnerabilities prioritized', 'information security', 'security policy', 'security requirements']
Retrieval Confidence: MEDIUM

RETRIEVED KNOWLEDGE:

[1] Vulnerability Management Policy
Department: Security
Source: vulnerability_management_policy.pdf
Similarity: 0.3253
Rerank Score: 0.3977
Content: Security vulnerabilities should be identified, assessed, prioritized, and remediated according to organizational risk management procedures. Critical vulnerabilities require expedited remediation. Vulnerability priority may depend on severity, exploitability, asset exposure, business impact, and whether the affected component is actively use

In [3]:
# ============================================================
# DAY 62/100 — ENTERPRISE AGENTIC RAG + VECTOR DATABASE
# PART 3 — AGENTIC RAG + GROUNDED GENERATION
# ============================================================
#
# CONTINUES FROM:
#
# PART 1:
#   Enterprise Documents
#        ↓
#   Chunking
#        ↓
#   Vector Representation
#        ↓
#   Vector DB
#
# PART 2:
#   User Query
#        ↓
#   Query Understanding
#        ↓
#   Agent Router
#        ↓
#   Query Expansion
#        ↓
#   Retrieval
#        ↓
#   Reranking
#        ↓
#   Retrieval Confidence
#
# PART 3:
#   Top-K Results
#        ↓
#   Context Construction
#        ↓
#   Grounded Generation
#        ↓
#   Claim Extraction
#        ↓
#   Grounding Validation
#        ↓
#   Source Citations
#        ↓
#   Safety / Fallback
#
# CPU + storage friendly.
# No large LLM download required.
# ============================================================


# ============================================================
# 1. IMPORTS
# ============================================================

import re
import time
import numpy as np
import pandas as pd

from collections import Counter


print("Part 3 libraries loaded successfully.")


# ============================================================
# 2. RAG CONFIGURATION
# ============================================================

AGENTIC_RAG_CONFIG = {

    "top_k": 5,

    "minimum_retrieval_confidence": "MEDIUM",

    "minimum_generation_score": 0.20,

    "grounding_threshold": 0.35,

    "citation_required": True,

    "safe_fallback_enabled": True,

    "max_context_chunks": 5,

    "max_context_characters": 5000,

    "max_answer_sentences": 5
}


print("\n============================================================")
print("AGENTIC RAG CONFIGURATION")
print("============================================================")

for key, value in AGENTIC_RAG_CONFIG.items():

    print(f"{key}: {value}")


# ============================================================
# 3. RAG RESPONSE TEMPLATE
# ============================================================
#
# This project uses a deterministic response generator.
#
# In production:
#
#     retrieved context
#            ↓
#        LLM prompt
#            ↓
#        Enterprise LLM
#            ↓
#        structured response
#
# The rest of the architecture remains the same.
# ============================================================


RAG_RESPONSE_TEMPLATE = {

    "answer": "",

    "confidence": "",

    "grounded": False,

    "citations": [],

    "sources": [],

    "retrieval_confidence": "",

    "retrieved_chunks": 0,

    "safe_fallback": False,

    "message": ""
}


# ============================================================
# 4. TEXT TOKENIZATION
# ============================================================

def rag_tokenize(text):

    """
    Lightweight tokenization used for grounding validation.
    """

    if text is None:

        return set()

    text = str(text).lower()

    tokens = re.findall(
        r"\b[a-zA-Z0-9]+\b",
        text
    )

    return set(tokens)


# ============================================================
# 5. STOPWORDS
# ============================================================

RAG_STOPWORDS = {
    "the",
    "a",
    "an",
    "is",
    "are",
    "was",
    "were",
    "be",
    "been",
    "being",
    "to",
    "of",
    "and",
    "or",
    "in",
    "on",
    "for",
    "with",
    "by",
    "from",
    "at",
    "as",
    "this",
    "that",
    "these",
    "those",
    "it",
    "they",
    "their",
    "them",
    "how",
    "what",
    "when",
    "where",
    "why",
    "which",
    "should",
    "can",
    "do",
    "does",
    "must",
    "may"
}


def meaningful_tokens(text):

    tokens = rag_tokenize(text)

    return {
        token
        for token in tokens
        if token not in RAG_STOPWORDS
        and len(token) > 2
    }


# ============================================================
# 6. CONTEXT CONSTRUCTION
# ============================================================
#
# Retrieved chunks are converted into a structured context.
#
# Example:
#
# [SOURCE 1]
# Title: Employee Leave Policy
# Department: HR
# Content: ...
#
# [SOURCE 2]
# Title: Remote Work Policy
# Department: HR
# Content: ...
#
# This is the context supplied to the generation layer.
# ============================================================

def build_rag_context(
    retrieval_response,
    max_chunks=None,
    max_characters=None
):

    if max_chunks is None:

        max_chunks = (
            AGENTIC_RAG_CONFIG[
                "max_context_chunks"
            ]
        )

    if max_characters is None:

        max_characters = (
            AGENTIC_RAG_CONFIG[
                "max_context_characters"
            ]
        )

    if not retrieval_response:

        return {
            "context": "",
            "sources": [],
            "chunk_count": 0,
            "characters": 0
        }

    results = retrieval_response.get(
        "results",
        []
    )

    results = results[:max_chunks]

    context_parts = []

    sources = []

    current_characters = 0

    for index, result in enumerate(
        results,
        start=1
    ):

        text = str(
            result.get(
                "text",
                ""
            )
        )

        title = str(
            result.get(
                "title",
                "Unknown"
            )
        )

        department = str(
            result.get(
                "department",
                "Unknown"
            )
        )

        source = str(
            result.get(
                "source",
                "Unknown"
            )
        )

        chunk_id = str(
            result.get(
                "chunk_id",
                "Unknown"
            )
        )

        block = (
            f"[SOURCE {index}]\n"
            f"Title: {title}\n"
            f"Department: {department}\n"
            f"Source: {source}\n"
            f"Chunk ID: {chunk_id}\n"
            f"Content: {text}\n"
        )

        if (
            current_characters
            + len(block)
            > max_characters
        ):

            break

        context_parts.append(
            block
        )

        sources.append({

            "citation_id":
                f"SOURCE {index}",

            "chunk_id":
                chunk_id,

            "document_id":
                result.get(
                    "document_id"
                ),

            "title":
                title,

            "department":
                department,

            "source":
                source,

            "similarity_score":
                result.get(
                    "similarity_score",
                    0.0
                ),

            "rerank_score":
                result.get(
                    "rerank_score",
                    result.get(
                        "similarity_score",
                        0.0
                    )
                )
        })

        current_characters += len(block)

    context = "\n".join(
        context_parts
    )

    return {

        "context":
            context,

        "sources":
            sources,

        "chunk_count":
            len(context_parts),

        "characters":
            len(context)
    }


# ============================================================
# 7. CONTEXT INSPECTION
# ============================================================

context_test_response = agentic_retrieval(
    "How are security vulnerabilities prioritized?"
)

context_test = build_rag_context(
    context_test_response
)

print("\n============================================================")
print("RAG CONTEXT")
print("============================================================")

print(
    context_test["context"]
)

print("\nContext chunks:")
print(
    context_test["chunk_count"]
)

print("\nContext characters:")
print(
    context_test["characters"]
)


# ============================================================
# 8. ANSWER EVIDENCE EXTRACTION
# ============================================================
#
# The lightweight generator extracts relevant sentences from
# retrieved documents.
#
# In a production LLM implementation, this function would be
# replaced by an LLM generation call.
# ============================================================

def split_sentences(text):

    if not text:

        return []

    sentences = re.split(
        r"(?<=[.!?])\s+",
        str(text).strip()
    )

    return [
        sentence.strip()
        for sentence in sentences
        if sentence.strip()
    ]


def score_sentence_against_query(
    sentence,
    query
):

    sentence_tokens = meaningful_tokens(
        sentence
    )

    query_tokens = meaningful_tokens(
        query
    )

    if not query_tokens:

        return 0.0

    overlap = (
        len(
            sentence_tokens
            & query_tokens
        )
        / len(query_tokens)
    )

    return overlap


# ============================================================
# 9. GROUNDED RESPONSE GENERATOR
# ============================================================
#
# Important principle:
#
# The generator is NOT allowed to invent information.
#
# It can only use retrieved context.
# ============================================================

def generate_grounded_answer(
    query,
    retrieval_response
):

    if not retrieval_response:

        return {
            "success": False,
            "answer": "",
            "reason": "No retrieval response."
        }

    retrieval_confidence = (
        retrieval_response.get(
            "retrieval_confidence",
            "LOW"
        )
    )

    results = retrieval_response.get(
        "results",
        []
    )

    # --------------------------------------------------------
    # Safety gate
    # --------------------------------------------------------

    if (
        retrieval_confidence == "LOW"
        or len(results) == 0
    ):

        return {

            "success": False,

            "answer": (
                "I could not find sufficiently "
                "relevant information in the "
                "available enterprise knowledge base."
            ),

            "reason":
                "Insufficient retrieval evidence.",

            "citations": [],

            "sources": [],

            "grounded": False,

            "safe_fallback": True
        }

    # --------------------------------------------------------
    # Build candidate sentences
    # --------------------------------------------------------

    sentence_candidates = []

    for result_index, result in enumerate(
        results,
        start=1
    ):

        text = str(
            result.get(
                "text",
                ""
            )
        )

        source_id = (
            f"SOURCE {result_index}"
        )

        sentences = split_sentences(
            text
        )

        for sentence in sentences:

            score = (
                score_sentence_against_query(
                    sentence,
                    query
                )
            )

            sentence_candidates.append({

                "sentence":
                    sentence,

                "score":
                    score,

                "source_id":
                    source_id,

                "source":
                    result.get(
                        "source"
                    ),

                "title":
                    result.get(
                        "title"
                    ),

                "chunk_id":
                    result.get(
                        "chunk_id"
                    )
            })

    # --------------------------------------------------------
    # Rank evidence
    # --------------------------------------------------------

    sentence_candidates = sorted(
        sentence_candidates,
        key=lambda x: x["score"],
        reverse=True
    )

    # --------------------------------------------------------
    # Select evidence
    # --------------------------------------------------------

    selected = []

    seen_sentences = set()

    for candidate in sentence_candidates:

        sentence_key = (
            candidate["sentence"]
            .lower()
            .strip()
        )

        if sentence_key in seen_sentences:

            continue

        if (
            candidate["score"]
            < AGENTIC_RAG_CONFIG[
                "minimum_generation_score"
            ]
        ):

            continue

        selected.append(
            candidate
        )

        seen_sentences.add(
            sentence_key
        )

        if (
            len(selected)
            >= AGENTIC_RAG_CONFIG[
                "max_answer_sentences"
            ]
        ):

            break

    # --------------------------------------------------------
    # No useful evidence
    # --------------------------------------------------------

    if not selected:

        return {

            "success": False,

            "answer": (
                "I could not find enough "
                "supporting information to "
                "answer this question reliably."
            ),

            "reason":
                "No sufficiently relevant evidence.",

            "citations": [],

            "sources": [],

            "grounded": False,

            "safe_fallback": True
        }

    # --------------------------------------------------------
    # Build answer
    # --------------------------------------------------------

    answer_parts = []

    citations = []

    sources = []

    for candidate in selected:

        answer_parts.append(
            candidate["sentence"]
        )

        citations.append(
            candidate["source_id"]
        )

        sources.append({

            "citation_id":
                candidate["source_id"],

            "title":
                candidate["title"],

            "source":
                candidate["source"],

            "chunk_id":
                candidate["chunk_id"]
        })

    answer = " ".join(
        answer_parts
    )

    # --------------------------------------------------------
    # Add source references
    # --------------------------------------------------------

    unique_citations = list(
        dict.fromkeys(
            citations
        )
    )

    if (
        AGENTIC_RAG_CONFIG[
            "citation_required"
        ]
    ):

        answer += (
            "\n\nSources: "
            + ", ".join(
                unique_citations
            )
        )

    return {

        "success": True,

        "answer":
            answer,

        "citations":
            unique_citations,

        "sources":
            sources,

        "grounded":
            True,

        "safe_fallback":
            False,

        "reason":
            "Answer generated from retrieved evidence."
    }


# ============================================================
# 10. GROUNDING VALIDATION
# ============================================================
#
# Goal:
#
# Determine whether the generated answer is supported by
# retrieved context.
#
# We compare meaningful answer tokens with context tokens.
#
# This is a lightweight validation layer.
#
# Production options:
#
# - NLI model
# - LLM-as-judge
# - claim verification
# - semantic entailment model
# - citation validation
# ============================================================

def grounding_score(
    answer,
    context
):

    answer_tokens = meaningful_tokens(
        answer
    )

    context_tokens = meaningful_tokens(
        context
    )

    if not answer_tokens:

        return 0.0

    if not context_tokens:

        return 0.0

    overlap = (
        len(
            answer_tokens
            & context_tokens
        )
        / len(answer_tokens)
    )

    return round(
        overlap,
        4
    )


def validate_grounding(
    answer,
    context,
    threshold=None
):

    if threshold is None:

        threshold = (
            AGENTIC_RAG_CONFIG[
                "grounding_threshold"
            ]
        )

    score = grounding_score(
        answer,
        context
    )

    grounded = (
        score >= threshold
    )

    return {

        "grounding_score":
            score,

        "threshold":
            threshold,

        "grounded":
            grounded,

        "status":
            "PASS"
            if grounded
            else "FAIL"
    }


# ============================================================
# 11. CLAIM EXTRACTION
# ============================================================
#
# Break answer into individual claims/sentences.
#
# This allows us to validate each claim against retrieved
# evidence.
# ============================================================

def extract_claims(answer):

    if not answer:

        return []

    # Remove source footer
    answer_without_sources = re.sub(
        r"\n\nSources:.*$",
        "",
        answer,
        flags=re.IGNORECASE |
        re.DOTALL
    )

    claims = split_sentences(
        answer_without_sources
    )

    return claims


# ============================================================
# 12. CLAIM-LEVEL GROUNDING
# ============================================================

def validate_claims(
    answer,
    context,
    threshold=None
):

    if threshold is None:

        threshold = (
            AGENTIC_RAG_CONFIG[
                "grounding_threshold"
            ]
        )

    claims = extract_claims(
        answer
    )

    validation_records = []

    for index, claim in enumerate(
        claims,
        start=1
    ):

        score = grounding_score(
            claim,
            context
        )

        validation_records.append({

            "claim_id":
                f"CLAIM-{index}",

            "claim":
                claim,

            "grounding_score":
                score,

            "grounded":
                score >= threshold
        })

    return pd.DataFrame(
        validation_records
    )


# ============================================================
# 13. SOURCE CITATION BUILDER
# ============================================================

def build_source_citations(
    retrieval_response
):

    if not retrieval_response:

        return []

    results = retrieval_response.get(
        "results",
        []
    )

    citations = []

    for index, result in enumerate(
        results,
        start=1
    ):

        citations.append({

            "citation_id":
                f"SOURCE {index}",

            "document_id":
                result.get(
                    "document_id"
                ),

            "chunk_id":
                result.get(
                    "chunk_id"
                ),

            "title":
                result.get(
                    "title"
                ),

            "department":
                result.get(
                    "department"
                ),

            "document_type":
                result.get(
                    "document_type"
                ),

            "source":
                result.get(
                    "source"
                ),

            "similarity_score":
                result.get(
                    "similarity_score",
                    0.0
                ),

            "rerank_score":
                result.get(
                    "rerank_score",
                    result.get(
                        "similarity_score",
                        0.0
                    )
                )
        })

    return citations


# ============================================================
# 14. SAFE RESPONSE HANDLER
# ============================================================

def safe_rag_response(
    query,
    retrieval_response,
    generation_response,
    grounding_response
):

    # --------------------------------------------------------
    # Case 1 — Retrieval failure
    # --------------------------------------------------------

    if not retrieval_response:

        return {

            "success": False,

            "answer": (
                "I could not retrieve relevant "
                "information from the knowledge base."
            ),

            "confidence": "LOW",

            "grounded": False,

            "citations": [],

            "safe_fallback": True
        }

    # --------------------------------------------------------
    # Case 2 — Generation failure
    # --------------------------------------------------------

    if not generation_response.get(
        "success",
        False
    ):

        return {

            "success": False,

            "answer":
                generation_response.get(
                    "answer",
                    "I could not generate a reliable answer."
                ),

            "confidence":
                retrieval_response.get(
                    "retrieval_confidence",
                    "LOW"
                ),

            "grounded": False,

            "citations": [],

            "safe_fallback": True
        }

    # --------------------------------------------------------
    # Case 3 — Grounding failure
    # --------------------------------------------------------

    if not grounding_response.get(
        "grounded",
        False
    ):

        return {

            "success": False,

            "answer": (
                "I found relevant information, "
                "but I could not sufficiently "
                "validate the generated response "
                "against the retrieved evidence."
            ),

            "confidence": "LOW",

            "grounded": False,

            "citations":
                build_source_citations(
                    retrieval_response
                ),

            "safe_fallback": True
        }

    # --------------------------------------------------------
    # Case 4 — Valid grounded response
    # --------------------------------------------------------

    retrieval_confidence = (
        retrieval_response.get(
            "retrieval_confidence",
            "LOW"
        )
    )

    if retrieval_confidence == "HIGH":

        confidence = "HIGH"

    elif retrieval_confidence == "MEDIUM":

        confidence = "MEDIUM"

    else:

        confidence = "LOW"

    return {

        "success": True,

        "answer":
            generation_response[
                "answer"
            ],

        "confidence":
            confidence,

        "grounded":
            True,

        "citations":
            build_source_citations(
                retrieval_response
            ),

        "safe_fallback":
            False
    }


# ============================================================
# 15. COMPLETE AGENTIC RAG PIPELINE
# ============================================================
#
# This is the central function for Part 3.
#
# User Query
#     ↓
# Agentic Retrieval
#     ↓
# Context Construction
#     ↓
# Grounded Generation
#     ↓
# Grounding Validation
#     ↓
# Claim Validation
#     ↓
# Citations
#     ↓
# Safe Response
# ============================================================

def agentic_rag(
    query
):

    pipeline_start = time.perf_counter()

    # --------------------------------------------------------
    # STEP 1 — Agentic Retrieval
    # --------------------------------------------------------

    retrieval_response = (
        agentic_retrieval(
            query=query,
            final_top_k=(
                AGENTIC_RAG_CONFIG[
                    "top_k"
                ]
            )
        )
    )

    # --------------------------------------------------------
    # STEP 2 — Context Construction
    # --------------------------------------------------------

    context_response = (
        build_rag_context(
            retrieval_response
        )
    )

    context = context_response[
        "context"
    ]

    # --------------------------------------------------------
    # STEP 3 — Grounded Generation
    # --------------------------------------------------------

    generation_response = (
        generate_grounded_answer(
            query=query,
            retrieval_response=
                retrieval_response
        )
    )

    # --------------------------------------------------------
    # STEP 4 — Grounding Validation
    # --------------------------------------------------------

    if generation_response.get(
        "success",
        False
    ):

        grounding_response = (
            validate_grounding(
                answer=
                    generation_response[
                        "answer"
                    ],

                context=context
            )
        )

    else:

        grounding_response = {

            "grounding_score": 0.0,

            "threshold":
                AGENTIC_RAG_CONFIG[
                    "grounding_threshold"
                ],

            "grounded": False,

            "status": "FAIL"
        }

    # --------------------------------------------------------
    # STEP 5 — Claim Validation
    # --------------------------------------------------------

    if generation_response.get(
        "success",
        False
    ):

        claim_validation_df = (
            validate_claims(
                answer=
                    generation_response[
                        "answer"
                    ],

                context=context
            )
        )

    else:

        claim_validation_df = (
            pd.DataFrame()
        )

    # --------------------------------------------------------
    # STEP 6 — Final Safety Gate
    # --------------------------------------------------------

    final_response = (
        safe_rag_response(

            query=query,

            retrieval_response=
                retrieval_response,

            generation_response=
                generation_response,

            grounding_response=
                grounding_response
        )
    )

    # --------------------------------------------------------
    # STEP 7 — Pipeline Metrics
    # --------------------------------------------------------

    total_latency_ms = (
        time.perf_counter()
        - pipeline_start
    ) * 1000

    if len(claim_validation_df) > 0:

        claim_grounding_rate = (
            claim_validation_df[
                "grounded"
            ].mean()
        )

    else:

        claim_grounding_rate = 0.0

    return {

        "query":
            query,

        "final_response":
            final_response,

        "retrieval":
            retrieval_response,

        "context":
            context_response,

        "generation":
            generation_response,

        "grounding":
            grounding_response,

        "claim_validation":
            claim_validation_df,

        "metrics": {

            "retrieved_chunks":
                context_response[
                    "chunk_count"
                ],

            "context_characters":
                context_response[
                    "characters"
                ],

            "grounding_score":
                grounding_response[
                    "grounding_score"
                ],

            "claim_grounding_rate":
                round(
                    claim_grounding_rate,
                    4
                ),

            "retrieval_confidence":
                retrieval_response.get(
                    "retrieval_confidence",
                    "LOW"
                ),

            "safe_fallback":
                final_response.get(
                    "safe_fallback",
                    False
                ),

            "total_latency_ms":
                round(
                    total_latency_ms,
                    3
                )
        }
    }


# ============================================================
# 16. END-TO-END AGENTIC RAG TEST
# ============================================================

rag_queries = [

    "How do employees request annual leave?",

    "How are security vulnerabilities prioritized?",

    "How should business expenses be reimbursed?",

    "What should be checked when processing an insurance claim?",

    "What are the standards for enterprise APIs?",

    "How should applications be deployed?"
]


print("\n============================================================")
print("END-TO-END AGENTIC RAG TEST")
print("============================================================")


rag_test_results = []


for query in rag_queries:

    response = agentic_rag(
        query
    )

    final_response = response[
        "final_response"
    ]

    metrics = response[
        "metrics"
    ]

    print("\n------------------------------------------------------------")

    print(
        "USER:",
        query
    )

    print(
        "\nANSWER:"
    )

    print(
        final_response[
            "answer"
        ]
    )

    print(
        "\nConfidence:",
        final_response[
            "confidence"
        ]
    )

    print(
        "Grounded:",
        final_response[
            "grounded"
        ]
    )

    print(
        "Citations:",
        len(
            final_response[
                "citations"
            ]
        )
    )

    print(
        "Grounding Score:",
        metrics[
            "grounding_score"
        ]
    )

    print(
        "Latency:",
        metrics[
            "total_latency_ms"
        ],
        "ms"
    )

    rag_test_results.append({

        "query":
            query,

        "success":
            final_response[
                "success"
            ],

        "confidence":
            final_response[
                "confidence"
            ],

        "grounded":
            final_response[
                "grounded"
            ],

        "citations":
            len(
                final_response[
                    "citations"
                ]
            ),

        "grounding_score":
            metrics[
                "grounding_score"
            ],

        "claim_grounding_rate":
            metrics[
                "claim_grounding_rate"
            ],

        "retrieved_chunks":
            metrics[
                "retrieved_chunks"
            ],

        "latency_ms":
            metrics[
                "total_latency_ms"
            ]
    })


rag_evaluation_df = pd.DataFrame(
    rag_test_results
)


# ============================================================
# 17. RAG EVALUATION TABLE
# ============================================================

print("\n============================================================")
print("RAG EVALUATION RESULTS")
print("============================================================")

display(
    rag_evaluation_df
)


# ============================================================
# 18. SOURCE CITATION INSPECTION
# ============================================================

citation_query = (
    "How are security vulnerabilities prioritized?"
)

citation_response = agentic_rag(
    citation_query
)

print("\n============================================================")
print("SOURCE CITATIONS")
print("============================================================")

display(
    pd.DataFrame(
        citation_response[
            "final_response"
        ][
            "citations"
        ]
    )
    if citation_response[
        "final_response"
    ][
        "citations"
    ]
    else pd.DataFrame()
)


# ============================================================
# 19. CLAIM VALIDATION INSPECTION
# ============================================================

print("\n============================================================")
print("CLAIM-LEVEL GROUNDING VALIDATION")
print("============================================================")

display(
    citation_response[
        "claim_validation"
    ]
)


# ============================================================
# 20. CONTEXT INSPECTION
# ============================================================

print("\n============================================================")
print("RETRIEVED CONTEXT USED FOR ANSWER")
print("============================================================")

print(
    citation_response[
        "context"
    ][
        "context"
    ]
)


# ============================================================
# 21. HALLUCINATION SAFETY TEST
# ============================================================
#
# Test with a question outside our knowledge base.
#
# Expected:
#
# LOW retrieval
#      ↓
# Safe fallback
#      ↓
# No confident answer
# ============================================================

hallucination_test_queries = [

    "What is the company policy for Mars colonization?",

    "How does the company insure spacecraft on Jupiter?",

    "What is the company's underwater city employee policy?"
]


hallucination_test_records = []


print("\n============================================================")
print("HALLUCINATION SAFETY TEST")
print("============================================================")


for query in hallucination_test_queries:

    response = agentic_rag(
        query
    )

    final_response = response[
        "final_response"
    ]

    metrics = response[
        "metrics"
    ]

    print("\nQuery:")
    print(query)

    print(
        "\nAnswer:"
    )

    print(
        final_response[
            "answer"
        ]
    )

    print(
        "\nConfidence:",
        final_response[
            "confidence"
        ]
    )

    print(
        "Grounded:",
        final_response[
            "grounded"
        ]
    )

    print(
        "Safe fallback:",
        final_response[
            "safe_fallback"
        ]
    )

    hallucination_test_records.append({

        "query":
            query,

        "confidence":
            final_response[
                "confidence"
            ],

        "grounded":
            final_response[
                "grounded"
            ],

        "safe_fallback":
            final_response[
                "safe_fallback"
            ],

        "grounding_score":
            metrics[
                "grounding_score"
            ]
    })


hallucination_safety_df = pd.DataFrame(
    hallucination_test_records
)


print("\n============================================================")
print("HALLUCINATION SAFETY RESULTS")
print("============================================================")

display(
    hallucination_safety_df
)


# ============================================================
# 22. CITATION COVERAGE
# ============================================================

successful_responses = rag_evaluation_df[
    rag_evaluation_df["success"] == True
]

if len(successful_responses) > 0:

    citation_coverage = (
        (
            successful_responses[
                "citations"
            ] > 0
        )
        .mean()
    )

else:

    citation_coverage = 0.0


# ============================================================
# 23. GROUNDING RATE
# ============================================================

if len(rag_evaluation_df) > 0:

    grounding_rate = (
        rag_evaluation_df[
            "grounded"
        ]
        .mean()
    )

else:

    grounding_rate = 0.0


# ============================================================
# 24. RAG SUCCESS RATE
# ============================================================

if len(rag_evaluation_df) > 0:

    rag_success_rate = (
        rag_evaluation_df[
            "success"
        ]
        .mean()
    )

else:

    rag_success_rate = 0.0


# ============================================================
# 25. SAFE FALLBACK RATE
# ============================================================

safe_fallback_rate = (
    hallucination_safety_df[
        "safe_fallback"
    ]
    .mean()
)


# ============================================================
# 26. AVERAGE GROUNDING SCORE
# ============================================================

average_grounding_score = (
    rag_evaluation_df[
        "grounding_score"
    ]
    .mean()
)


# ============================================================
# 27. AVERAGE LATENCY
# ============================================================

average_rag_latency = (
    rag_evaluation_df[
        "latency_ms"
    ]
    .mean()
)


# ============================================================
# 28. PART 3 SCORECARD
# ============================================================

PART_3_SCORECARD = {

    "rag_success_rate_percent":
        round(
            rag_success_rate * 100,
            2
        ),

    "grounding_rate_percent":
        round(
            grounding_rate * 100,
            2
        ),

    "citation_coverage_percent":
        round(
            citation_coverage * 100,
            2
        ),

    "average_grounding_score":
        round(
            average_grounding_score,
            4
        ),

    "safe_fallback_rate_percent":
        round(
            safe_fallback_rate * 100,
            2
        ),

    "average_rag_latency_ms":
        round(
            average_rag_latency,
            3
        ),

    "queries_evaluated":
        len(
            rag_evaluation_df
        )
}


print("\n============================================================")
print("PART 3 RAG SCORECARD")
print("============================================================")


for metric, value in PART_3_SCORECARD.items():

    print(
        f"{metric}: {value}"
    )


# ============================================================
# 29. FULL PIPELINE TRACE
# ============================================================

def show_full_rag_trace(
    query
):

    response = agentic_rag(
        query
    )

    retrieval = response[
        "retrieval"
    ]

    context = response[
        "context"
    ]

    generation = response[
        "generation"
    ]

    grounding = response[
        "grounding"
    ]

    final = response[
        "final_response"
    ]

    print("\n============================================================")
    print("FULL AGENTIC RAG TRACE")
    print("============================================================")

    print(
        "\n1. USER QUERY"
    )

    print(query)

    print(
        "\n2. AGENT DECISION"
    )

    print(
        "Domain:",
        retrieval[
            "detected_domain"
        ]
    )

    print(
        "Tool:",
        retrieval[
            "selected_tool"
        ]
    )

    print(
        "Filters:",
        retrieval[
            "metadata_filters"
        ]
    )

    print(
        "\n3. QUERY VARIANTS"
    )

    for variant in retrieval[
        "query_variants"
    ]:

        print(
            "→",
            variant
        )

    print(
        "\n4. RETRIEVAL"
    )

    print(
        "Candidates:",
        retrieval[
            "candidate_count"
        ]
    )

    print(
        "Final results:",
        retrieval[
            "final_count"
        ]
    )

    print(
        "Retrieval confidence:",
        retrieval[
            "retrieval_confidence"
        ]
    )

    print(
        "\n5. CONTEXT"
    )

    print(
        "Chunks:",
        context[
            "chunk_count"
        ]
    )

    print(
        "Characters:",
        context[
            "characters"
        ]
    )

    print(
        "\n6. GENERATION"
    )

    print(
        generation[
            "answer"
        ]
    )

    print(
        "\n7. GROUNDING VALIDATION"
    )

    print(
        "Score:",
        grounding[
            "grounding_score"
        ]
    )

    print(
        "Status:",
        grounding[
            "status"
        ]
    )

    print(
        "\n8. FINAL RESPONSE"
    )

    print(
        final[
            "answer"
        ]
    )

    print(
        "\n9. FINAL STATUS"
    )

    print(
        "Confidence:",
        final[
            "confidence"
        ]
    )

    print(
        "Grounded:",
        final[
            "grounded"
        ]
    )

    print(
        "Safe fallback:",
        final[
            "safe_fallback"
        ]
    )

    return response


# Demonstration
full_trace_response = show_full_rag_trace(
    "How are security vulnerabilities prioritized?"
)


# ============================================================
# 30. PRODUCTION LLM REPLACEMENT INTERFACE
# ============================================================
#
# This function demonstrates where a real LLM would be plugged
# into the architecture.
#
# The rest of the pipeline does NOT need to change.
#
# Production:
#
# context
#    ↓
# prompt
#    ↓
# LLM
#    ↓
# answer
#    ↓
# grounding validator
# ============================================================

def production_llm_generation_interface(
    query,
    context,
    system_instruction=None
):

    """
    Placeholder interface for a production LLM.

    This intentionally does not call an external model.

    Production implementation could connect to:
        - OpenAI
        - Azure OpenAI
        - Anthropic
        - Gemini
        - Azure AI Foundry
        - Bedrock
        - Local Llama/Qwen models

    The important architectural interface is:

        query + context
                ↓
             LLM
                ↓
           generated answer
    """

    if system_instruction is None:

        system_instruction = (
            "Answer only using the supplied context. "
            "Do not invent unsupported information."
        )

    prompt = f"""
SYSTEM:
{system_instruction}

USER QUERY:
{query}

RETRIEVED CONTEXT:
{context}

INSTRUCTIONS:
1. Use only the retrieved context.
2. Do not invent facts.
3. If the context does not contain enough information,
   explicitly say that sufficient evidence was not found.
4. Provide source references when possible.
"""

    return {
        "prompt": prompt,
        "ready_for_llm": True
    }


# ============================================================
# 31. PRODUCTION PROMPT DEMONSTRATION
# ============================================================

production_prompt_demo = (
    production_llm_generation_interface(

        query=
            "How are security vulnerabilities prioritized?",

        context=
            full_trace_response[
                "context"
            ][
                "context"
            ]
    )
)


print("\n============================================================")
print("PRODUCTION LLM INTERFACE")
print("============================================================")

print(
    production_prompt_demo[
        "prompt"
    ]
)


# ============================================================
# 32. FINAL PART 3 VALIDATION
# ============================================================

part_3_validation = {

    "Context construction":
        context_test[
            "chunk_count"
        ] > 0,

    "Context contains source metadata":
        "Source:" in context_test[
            "context"
        ],

    "Grounded generation":
        citation_response[
            "generation"
        ].get(
            "success",
            False
        ),

    "Grounding validation":
        "grounding_score"
        in citation_response[
            "grounding"
        ],

    "Claim extraction":
        len(
            extract_claims(
                citation_response[
                    "generation"
                ].get(
                    "answer",
                    ""
                )
            )
        ) >= 0,

    "Source citations":
        len(
            citation_response[
                "final_response"
            ][
                "citations"
            ]
        ) > 0,

    "End-to-end RAG":
        citation_response[
            "final_response"
        ][
            "success"
        ],

    "RAG evaluation":
        len(
            rag_evaluation_df
        ) > 0,

    "Safe fallback":
        len(
            hallucination_safety_df
        ) > 0,

    "Production LLM interface":
        production_prompt_demo[
            "ready_for_llm"
        ]
}


print("\n============================================================")
print("PART 3 VALIDATION")
print("============================================================")


all_part_3_passed = True


for check, status in part_3_validation.items():

    print(
        f"{'PASS' if status else 'FAIL'} | {check}"
    )

    if not status:

        all_part_3_passed = False


print("\nOverall Part 3 Status:")


if all_part_3_passed:

    print(
        "✅ PART 3 COMPLETED SUCCESSFULLY"
    )

else:

    print(
        "❌ PART 3 REQUIRES ATTENTION"
    )


# ============================================================
# 33. IMPORTANT VARIABLES CREATED
# ============================================================

print("\n============================================================")
print("IMPORTANT VARIABLES CREATED IN PART 3")
print("============================================================")


part_3_variables = [

    "AGENTIC_RAG_CONFIG",

    "RAG_RESPONSE_TEMPLATE",

    "RAG_STOPWORDS",

    "build_rag_context",

    "rag_tokenize",

    "meaningful_tokens",

    "split_sentences",

    "score_sentence_against_query",

    "generate_grounded_answer",

    "grounding_score",

    "validate_grounding",

    "extract_claims",

    "validate_claims",

    "build_source_citations",

    "safe_rag_response",

    "agentic_rag",

    "rag_test_results",

    "rag_evaluation_df",

    "hallucination_safety_df",

    "PART_3_SCORECARD",

    "show_full_rag_trace",

    "production_llm_generation_interface",

    "production_prompt_demo",

    "full_trace_response"
]


for variable in part_3_variables:

    print(
        "→",
        variable
    )


# ============================================================
# 34. FINAL PART 3 ARCHITECTURE
# ============================================================

print("""
============================================================
DAY 62 — PART 3 FINAL ARCHITECTURE
============================================================

                       USER
                        ↓
                  User Query
                        ↓
                Agentic Retrieval
                        ↓
             Query Transformation
                        ↓
                Query Expansion
                        ↓
               Vector Retrieval
                        ↓
                Metadata Filter
                        ↓
                  Reranking
                        ↓
                    Top-K
                        ↓
             Retrieval Confidence
                        ↓
              Context Construction
                        ↓
               Grounded Generation
                        ↓
               Claim Extraction
                        ↓
             Grounding Validation
                        ↓
                ┌───────┴───────┐
                ↓               ↓
              PASS             FAIL
                ↓               ↓
          Source Citations    Safe Fallback
                ↓
          Final Response
                        ↓
                    User


============================================================
PART 3 COMPONENTS
============================================================

1. Agentic Retrieval
2. Context Construction
3. Grounded Generation
4. Claim Extraction
5. Grounding Validation
6. Citation Management
7. Safe Fallback
8. Hallucination Safety
9. RAG Evaluation
10. Production LLM Interface


============================================================
PART 3 COMPLETE
============================================================
""")


# ============================================================
# 35. INTERVIEW-READY EXPLANATION
# ============================================================

print("""
============================================================
INTERVIEW-READY EXPLANATION
============================================================

"For Part 3, I converted the agentic retrieval system into a
grounded RAG pipeline.

The agent first retrieves and reranks the most relevant
enterprise chunks. I then construct a structured context from
those chunks while preserving source metadata such as document
ID, title, department, source and chunk ID.

The generation layer is designed to answer only from the
retrieved context. Since the development environment was
resource constrained, I used a lightweight deterministic
generator, but I designed a clean LLM interface so it can be
replaced by an enterprise LLM without changing the retrieval
architecture.

After generation, I perform grounding validation by comparing
the generated answer against the retrieved evidence. I also
perform claim-level validation to identify whether individual
claims are supported.

Finally, I attach source citations and apply a safety gate. If
retrieval evidence is insufficient or the generated response
cannot be adequately grounded, the system falls back instead
of returning an unsupported answer.

I also evaluated the pipeline using RAG success rate, grounding
rate, citation coverage, grounding score, safe fallback rate
and latency.

This gives me an end-to-end agentic RAG pipeline where retrieval,
generation, grounding and source attribution are separate
observable components."
============================================================
""")

Part 3 libraries loaded successfully.

AGENTIC RAG CONFIGURATION
top_k: 5
minimum_retrieval_confidence: MEDIUM
minimum_generation_score: 0.2
grounding_threshold: 0.35
citation_required: True
safe_fallback_enabled: True
max_context_chunks: 5
max_context_characters: 5000
max_answer_sentences: 5

RAG CONTEXT
[SOURCE 1]
Title: Vulnerability Management Policy
Department: Security
Source: vulnerability_management_policy.pdf
Chunk ID: SEC-002-CHUNK-0
Content: Security vulnerabilities should be identified, assessed, prioritized, and remediated according to organizational risk management procedures. Critical vulnerabilities require expedited remediation. Vulnerability priority may depend on severity, exploitability, asset exposure, business impact, and whether the affected component is actively used.

[SOURCE 2]
Title: Information Security Policy
Department: Security
Source: information_security_policy.pdf
Chunk ID: SEC-001-CHUNK-0
Content: Employees must protect confidential company informatio

,query,success,confidence,grounded,citations,grounding_score,claim_grounding_rate,retrieved_chunks,latency_ms
0,How do employees request annual leave?,True,MEDIUM,True,2,0.9730,1.0,2,20.060
1,How are security vulnerabilities prioritized?,True,MEDIUM,True,2,0.9706,1.0,2,19.015
2,How should business expenses be reimbursed?,True,MEDIUM,True,2,0.9655,1.0,2,19.291
3,What should be checked when processing an insu...,True,MEDIUM,True,2,0.9688,1.0,2,19.017
4,What are the standards for enterprise APIs?,True,MEDIUM,True,2,0.9630,1.0,2,18.706
5,How should applications be deployed?,True,MEDIUM,True,2,0.8571,1.0,2,7.985



SOURCE CITATIONS


,citation_id,document_id,chunk_id,title,department,document_type,source,similarity_score,rerank_score
0,SOURCE 1,SEC-002,SEC-002-CHUNK-0,Vulnerability Management Policy,Security,Security Policy,vulnerability_management_policy.pdf,0.3253,0.3977
1,SOURCE 2,SEC-001,SEC-001-CHUNK-0,Information Security Policy,Security,Security Policy,information_security_policy.pdf,0.1176,0.1823



CLAIM-LEVEL GROUNDING VALIDATION


,claim_id,claim,grounding_score,grounded
0,CLAIM-1,"Security vulnerabilities should be identified,...",1.0,True
1,CLAIM-2,Critical vulnerabilities require expedited rem...,1.0,True
2,CLAIM-3,Employees must protect confidential company in...,1.0,True
3,CLAIM-4,"Security incidents, suspicious activity, and p...",1.0,True



RETRIEVED CONTEXT USED FOR ANSWER
[SOURCE 1]
Title: Vulnerability Management Policy
Department: Security
Source: vulnerability_management_policy.pdf
Chunk ID: SEC-002-CHUNK-0
Content: Security vulnerabilities should be identified, assessed, prioritized, and remediated according to organizational risk management procedures. Critical vulnerabilities require expedited remediation. Vulnerability priority may depend on severity, exploitability, asset exposure, business impact, and whether the affected component is actively used.

[SOURCE 2]
Title: Information Security Policy
Department: Security
Source: information_security_policy.pdf
Chunk ID: SEC-001-CHUNK-0
Content: Employees must protect confidential company information and follow approved security controls. Passwords must not be shared with other users. Multi-factor authentication should be enabled for supported applications. Security incidents, suspicious activity, and potential data exposure must be reported through the approved sec

,query,confidence,grounded,safe_fallback,grounding_score
0,What is the company policy for Mars colonization?,MEDIUM,True,False,0.9737
1,How does the company insure spacecraft on Jupi...,LOW,False,True,0.0000
2,What is the company's underwater city employee...,MEDIUM,True,False,0.9714



PART 3 RAG SCORECARD
rag_success_rate_percent: 100.0
grounding_rate_percent: 100.0
citation_coverage_percent: 100.0
average_grounding_score: 0.9497
safe_fallback_rate_percent: 33.33
average_rag_latency_ms: 17.346
queries_evaluated: 6

FULL AGENTIC RAG TRACE

1. USER QUERY
How are security vulnerabilities prioritized?

2. AGENT DECISION
Domain: Security
Tool: vector_search
Filters: {'department': 'Security'}

3. QUERY VARIANTS
→ how are security vulnerabilities prioritized
→ information security
→ security policy
→ security requirements

4. RETRIEVAL
Candidates: 2
Final results: 2
Retrieval confidence: MEDIUM

5. CONTEXT
Chunks: 2
Characters: 980

6. GENERATION
Security vulnerabilities should be identified, assessed, prioritized, and remediated according to organizational risk management procedures. Critical vulnerabilities require expedited remediation. Employees must protect confidential company information and follow approved security controls. Security incidents, suspicious activit

In [4]:
# ============================================================
# DAY 62/100 — ENTERPRISE AGENTIC RAG SYSTEM
# PART 4 — PRODUCTION API + MONITORING + DEPLOYMENT
# ============================================================
#
# CONTINUES FROM PART 1 + PART 2 + PART 3
#
# PART 1
#   Documents
#      ↓
#   Chunking
#      ↓
#   Vector Representation
#      ↓
#   Vector DB
#
# PART 2
#   Query
#      ↓
#   Agent Router
#      ↓
#   Query Expansion
#      ↓
#   Retrieval
#      ↓
#   Reranking
#      ↓
#   Confidence
#
# PART 3
#   Top-K Context
#      ↓
#   Grounded Generation
#      ↓
#   Grounding Validation
#      ↓
#   Citations
#      ↓
#   Safe Fallback
#
# PART 4
#   FastAPI
#      ↓
#   Request Validation
#      ↓
#   Session Management
#      ↓
#   Agentic RAG
#      ↓
#   Monitoring
#      ↓
#   Evaluation
#      ↓
#   Docker
#      ↓
#   Cloud-Ready Architecture
#
# CPU + STORAGE FRIENDLY
# No large model download.
# ============================================================


# ============================================================
# 1. INSTALL / IMPORT DEPENDENCIES
# ============================================================

# If FastAPI is already installed, these commands are harmless
# when commented out.
#
# Uncomment only if your notebook environment needs them:
#
# !pip install -q fastapi uvicorn httpx pydantic


import time
import uuid
import logging
import statistics
import os
import json

from datetime import datetime, timezone
from typing import Optional, Dict, List, Any

import pandas as pd
import numpy as np

from fastapi import (
    FastAPI,
    HTTPException,
    Request
)

from fastapi.responses import JSONResponse

from pydantic import BaseModel, Field


print("Part 4 libraries loaded successfully.")


# ============================================================
# 2. VERIFY PART 3 DEPENDENCY
# ============================================================
#
# Part 4 depends primarily on:
#
#     agentic_rag()
#
# from Part 3.
#
# We intentionally do not recreate Parts 1–3.
# ============================================================

if "agentic_rag" not in globals():

    raise RuntimeError(
        "agentic_rag() was not found. "
        "Run Parts 1, 2 and 3 before running Part 4."
    )


print("✅ Part 3 agentic_rag() detected.")


# ============================================================
# 3. PRODUCTION CONFIGURATION
# ============================================================

PRODUCTION_CONFIG = {

    "application_name":
        "Enterprise Agentic RAG Assistant",

    "version":
        "1.0.0",

    "environment":
        "development",

    "api_prefix":
        "/api/v1",

    "default_top_k":
        5,

    "max_query_length":
        1000,

    "max_session_history":
        10,

    "request_timeout_seconds":
        30,

    "enable_citations":
        True,

    "enable_grounding_validation":
        True,

    "enable_safe_fallback":
        True,

    "monitoring_enabled":
        True
}


print("\n============================================================")
print("PRODUCTION CONFIGURATION")
print("============================================================")

for key, value in PRODUCTION_CONFIG.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# 4. LOGGING
# ============================================================

logging.basicConfig(
    level=logging.INFO,
    format=(
        "%(asctime)s | "
        "%(levelname)s | "
        "%(message)s"
    )
)

logger = logging.getLogger(
    "enterprise_agentic_rag"
)


logger.info(
    "Enterprise Agentic RAG service initialized."
)


# ============================================================
# 5. APPLICATION METRICS
# ============================================================
#
# Notebook/demo metrics.
#
# In real production:
#
#     Prometheus
#          ↓
#     Grafana
#
# would normally be used.
# ============================================================

class ApplicationMetrics:

    def __init__(self):

        self.total_requests = 0

        self.successful_requests = 0

        self.failed_requests = 0

        self.total_latency_ms = 0.0

        self.latencies_ms = []

        self.rag_successes = 0

        self.rag_failures = 0

        self.grounded_responses = 0

        self.ungrounded_responses = 0

        self.safe_fallbacks = 0

        self.total_citations = 0

        self.total_retrieved_chunks = 0

        self.started_at = datetime.now(
            timezone.utc
        ).isoformat()

    def record_request(
        self,
        latency_ms,
        success=True
    ):

        self.total_requests += 1

        self.total_latency_ms += latency_ms

        self.latencies_ms.append(
            latency_ms
        )

        if success:

            self.successful_requests += 1

        else:

            self.failed_requests += 1

    def record_rag(
        self,
        response
    ):

        if response.get(
            "success",
            False
        ):

            self.rag_successes += 1

        else:

            self.rag_failures += 1

        if response.get(
            "grounded",
            False
        ):

            self.grounded_responses += 1

        else:

            self.ungrounded_responses += 1

        if response.get(
            "safe_fallback",
            False
        ):

            self.safe_fallbacks += 1

        self.total_citations += len(
            response.get(
                "citations",
                []
            )
        )

    def average_latency(self):

        if not self.latencies_ms:

            return 0.0

        return statistics.mean(
            self.latencies_ms
        )

    def percentile(
        self,
        percentile_value
    ):

        if not self.latencies_ms:

            return 0.0

        return float(
            np.percentile(
                self.latencies_ms,
                percentile_value
            )
        )

    def success_rate(self):

        if self.total_requests == 0:

            return 0.0

        return (
            self.successful_requests
            / self.total_requests
        )

    def grounding_rate(self):

        total = (
            self.grounded_responses
            + self.ungrounded_responses
        )

        if total == 0:

            return 0.0

        return (
            self.grounded_responses
            / total
        )

    def fallback_rate(self):

        if self.total_requests == 0:

            return 0.0

        return (
            self.safe_fallbacks
            / self.total_requests
        )

    def snapshot(self):

        return {

            "total_requests":
                self.total_requests,

            "successful_requests":
                self.successful_requests,

            "failed_requests":
                self.failed_requests,

            "success_rate_percent":
                round(
                    self.success_rate() * 100,
                    2
                ),

            "average_latency_ms":
                round(
                    self.average_latency(),
                    3
                ),

            "p50_latency_ms":
                round(
                    self.percentile(50),
                    3
                ),

            "p95_latency_ms":
                round(
                    self.percentile(95),
                    3
                ),

            "p99_latency_ms":
                round(
                    self.percentile(99),
                    3
                ),

            "rag_successes":
                self.rag_successes,

            "rag_failures":
                self.rag_failures,

            "grounded_responses":
                self.grounded_responses,

            "ungrounded_responses":
                self.ungrounded_responses,

            "grounding_rate_percent":
                round(
                    self.grounding_rate() * 100,
                    2
                ),

            "safe_fallbacks":
                self.safe_fallbacks,

            "safe_fallback_rate_percent":
                round(
                    self.fallback_rate() * 100,
                    2
                ),

            "total_citations":
                self.total_citations,

            "started_at":
                self.started_at
        }


metrics = ApplicationMetrics()


# ============================================================
# 6. SESSION STORE
# ============================================================
#
# Demo implementation:
#
#     Python dictionary
#
# Production implementation:
#
#     Redis
#     PostgreSQL
#     Cosmos DB
#     DynamoDB
#     etc.
# ============================================================

SESSION_STORE = {}


def create_session():

    session_id = str(
        uuid.uuid4()
    )

    SESSION_STORE[
        session_id
    ] = {

        "session_id":
            session_id,

        "created_at":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "updated_at":
            datetime.now(
                timezone.utc
            ).isoformat(),

        "messages": []
    }

    return session_id


def get_session(
    session_id
):

    return SESSION_STORE.get(
        session_id
    )


def delete_session(
    session_id
):

    if session_id in SESSION_STORE:

        del SESSION_STORE[
            session_id
        ]

        return True

    return False


def add_session_message(
    session_id,
    role,
    content
):

    session = get_session(
        session_id
    )

    if session is None:

        return False

    session[
        "messages"
    ].append({

        "role":
            role,

        "content":
            content,

        "timestamp":
            datetime.now(
                timezone.utc
            ).isoformat()
    })

    # Keep only latest messages
    # for lightweight memory.

    max_messages = (
        PRODUCTION_CONFIG[
            "max_session_history"
        ] * 2
    )

    session[
        "messages"
    ] = session[
        "messages"
    ][
        -max_messages:
    ]

    session[
        "updated_at"
    ] = datetime.now(
        timezone.utc
    ).isoformat()

    return True


# ============================================================
# 7. SESSION-AWARE QUERY BUILDER
# ============================================================

def build_session_context(
    session_id
):

    session = get_session(
        session_id
    )

    if session is None:

        return ""

    messages = session[
        "messages"
    ]

    if not messages:

        return ""

    recent_messages = messages[
        -PRODUCTION_CONFIG[
            "max_session_history"
        ]:
        ]

    context_lines = []

    for message in recent_messages:

        role = message[
            "role"
        ].upper()

        content = message[
            "content"
        ]

        context_lines.append(
            f"{role}: {content}"
        )

    return "\n".join(
        context_lines
    )


# ============================================================
# 8. REQUEST / RESPONSE MODELS
# ============================================================

class ChatRequest(BaseModel):

    query: str = Field(
        ...,
        min_length=1,
        max_length=1000
    )

    session_id: Optional[str] = None


class ChatResponse(BaseModel):

    request_id: str

    session_id: str

    query: str

    answer: str

    success: bool

    confidence: str

    grounded: bool

    safe_fallback: bool

    citations: List[Dict[str, Any]]

    retrieval_confidence: str

    retrieved_chunks: int

    grounding_score: float

    latency_ms: float


class SearchRequest(BaseModel):

    query: str = Field(
        ...,
        min_length=1,
        max_length=1000
    )


class SessionCreateResponse(BaseModel):

    session_id: str

    created_at: str


class HealthResponse(BaseModel):

    status: str

    service: str

    version: str

    timestamp: str


# ============================================================
# 9. FASTAPI APPLICATION
# ============================================================

app = FastAPI(

    title=
        PRODUCTION_CONFIG[
            "application_name"
        ],

    version=
        PRODUCTION_CONFIG[
            "version"
        ],

    description=(
        "Production-style Enterprise "
        "Agentic RAG API"
    )
)


# ============================================================
# 10. REQUEST MONITORING MIDDLEWARE
# ============================================================

@app.middleware(
    "http"
)
async def monitoring_middleware(
    request: Request,
    call_next
):

    request_id = str(
        uuid.uuid4()
    )

    start_time = time.perf_counter()

    try:

        response = await call_next(
            request
        )

        success = (
            response.status_code < 400
        )

    except Exception:

        success = False

        raise

    finally:

        latency_ms = (
            time.perf_counter()
            - start_time
        ) * 1000

        metrics.record_request(
            latency_ms=latency_ms,
            success=success
        )

        logger.info(
            (
                "request_id=%s "
                "method=%s "
                "path=%s "
                "latency_ms=%.3f "
                "success=%s"
            ),
            request_id,
            request.method,
            request.url.path,
            latency_ms,
            success
        )

    response.headers[
        "X-Request-ID"
    ] = request_id

    return response


# ============================================================
# 11. ROOT ENDPOINT
# ============================================================

@app.get("/")
def root():

    return {

        "service":
            PRODUCTION_CONFIG[
                "application_name"
            ],

        "version":
            PRODUCTION_CONFIG[
                "version"
            ],

        "status":
            "running",

        "message":
            "Enterprise Agentic RAG API"
    }


# ============================================================
# 12. HEALTH ENDPOINT
# ============================================================

@app.get(
    "/health",
    response_model=HealthResponse
)
def health():

    return {

        "status":
            "healthy",

        "service":
            PRODUCTION_CONFIG[
                "application_name"
            ],

        "version":
            PRODUCTION_CONFIG[
                "version"
            ],

        "timestamp":
            datetime.now(
                timezone.utc
            ).isoformat()
    }


# ============================================================
# 13. INFO ENDPOINT
# ============================================================

@app.get(
    "/api/v1/info"
)
def application_info():

    return {

        "application":
            PRODUCTION_CONFIG[
                "application_name"
            ],

        "version":
            PRODUCTION_CONFIG[
                "version"
            ],

        "environment":
            PRODUCTION_CONFIG[
                "environment"
            ],

        "features": [

            "Agentic Retrieval",

            "Query Expansion",

            "Vector Search",

            "Metadata Filtering",

            "Reranking",

            "Retrieval Confidence",

            "RAG",

            "Grounding Validation",

            "Source Citations",

            "Safe Fallback",

            "Session Management",

            "Monitoring"
        ]
    }


# ============================================================
# 14. METRICS ENDPOINT
# ============================================================

@app.get(
    "/api/v1/metrics"
)
def application_metrics():

    return metrics.snapshot()


# ============================================================
# 15. SESSION CREATE ENDPOINT
# ============================================================

@app.post(
    "/api/v1/session",
    response_model=SessionCreateResponse
)
def create_session_endpoint():

    session_id = create_session()

    session = get_session(
        session_id
    )

    return {

        "session_id":
            session_id,

        "created_at":
            session[
                "created_at"
            ]
    }


# ============================================================
# 16. SESSION GET ENDPOINT
# ============================================================

@app.get(
    "/api/v1/session/{session_id}"
)
def get_session_endpoint(
    session_id: str
):

    session = get_session(
        session_id
    )

    if session is None:

        raise HTTPException(

            status_code=404,

            detail=
                "Session not found."
        )

    return session


# ============================================================
# 17. SESSION DELETE ENDPOINT
# ============================================================

@app.delete(
    "/api/v1/session/{session_id}"
)
def delete_session_endpoint(
    session_id: str
):

    deleted = delete_session(
        session_id
    )

    if not deleted:

        raise HTTPException(

            status_code=404,

            detail=
                "Session not found."
        )

    return {

        "success":
            True,

        "session_id":
            session_id,

        "message":
            "Session deleted."
    }


# ============================================================
# 18. SEARCH ENDPOINT
# ============================================================
#
# Search endpoint returns retrieval information without
# generating an answer.
# ============================================================

@app.post(
    "/api/v1/search"
)
def search_endpoint(
    request: SearchRequest
):

    start_time = time.perf_counter()

    try:

        result = agentic_rag(
            request.query
        )

        retrieval = result[
            "retrieval"
        ]

        latency_ms = (
            time.perf_counter()
            - start_time
        ) * 1000

        return {

            "success":
                True,

            "query":
                request.query,

            "retrieval_confidence":
                retrieval.get(
                    "retrieval_confidence",
                    "LOW"
                ),

            "detected_domain":
                retrieval.get(
                    "detected_domain"
                ),

            "selected_tool":
                retrieval.get(
                    "selected_tool"
                ),

            "results":
                retrieval.get(
                    "results",
                    []
                ),

            "candidate_count":
                retrieval.get(
                    "candidate_count",
                    0
                ),

            "final_count":
                retrieval.get(
                    "final_count",
                    0
                ),

            "latency_ms":
                round(
                    latency_ms,
                    3
                )
        }

    except Exception as error:

        logger.exception(
            "Search endpoint failed."
        )

        raise HTTPException(

            status_code=500,

            detail=(
                "Search processing failed: "
                + str(error)
            )
        )


# ============================================================
# 19. CHAT / RAG ENDPOINT
# ============================================================

@app.post(
    "/api/v1/chat",
    response_model=ChatResponse
)
def chat_endpoint(
    request: ChatRequest
):

    request_id = str(
        uuid.uuid4()
    )

    start_time = time.perf_counter()

    # --------------------------------------------------------
    # SESSION
    # --------------------------------------------------------

    session_id = (
        request.session_id
        if request.session_id
        else create_session()
    )

    if get_session(
        session_id
    ) is None:

        raise HTTPException(

            status_code=404,

            detail=
                "Session not found."
        )

    # --------------------------------------------------------
    # STORE USER MESSAGE
    # --------------------------------------------------------

    add_session_message(

        session_id=session_id,

        role="user",

        content=request.query
    )

    # --------------------------------------------------------
    # RUN AGENTIC RAG
    # --------------------------------------------------------

    try:

        rag_result = agentic_rag(
            request.query
        )

        final_response = rag_result[
            "final_response"
        ]

        rag_metrics = rag_result[
            "metrics"
        ]

        metrics.record_rag(
            final_response
        )

    except Exception as error:

        logger.exception(
            "RAG pipeline failed."
        )

        raise HTTPException(

            status_code=500,

            detail=(
                "RAG pipeline failed: "
                + str(error)
            )
        )

    # --------------------------------------------------------
    # STORE ASSISTANT RESPONSE
    # --------------------------------------------------------

    answer = final_response.get(
        "answer",
        ""
    )

    add_session_message(

        session_id=session_id,

        role="assistant",

        content=answer
    )

    # --------------------------------------------------------
    # LATENCY
    # --------------------------------------------------------

    latency_ms = (
        time.perf_counter()
        - start_time
    ) * 1000

    # --------------------------------------------------------
    # RESPONSE
    # --------------------------------------------------------

    return {

        "request_id":
            request_id,

        "session_id":
            session_id,

        "query":
            request.query,

        "answer":
            answer,

        "success":
            final_response.get(
                "success",
                False
            ),

        "confidence":
            final_response.get(
                "confidence",
                "LOW"
            ),

        "grounded":
            final_response.get(
                "grounded",
                False
            ),

        "safe_fallback":
            final_response.get(
                "safe_fallback",
                False
            ),

        "citations":
            final_response.get(
                "citations",
                []
            ),

        "retrieval_confidence":
            rag_metrics.get(
                "retrieval_confidence",
                "LOW"
            ),

        "retrieved_chunks":
            rag_metrics.get(
                "retrieved_chunks",
                0
            ),

        "grounding_score":
            rag_metrics.get(
                "grounding_score",
                0.0
            ),

        "latency_ms":
            round(
                latency_ms,
                3
            )
    }


# ============================================================
# 20. GLOBAL EXCEPTION HANDLER
# ============================================================

@app.exception_handler(
    Exception
)
async def global_exception_handler(
    request: Request,
    exc: Exception
):

    logger.exception(
        "Unhandled application exception."
    )

    return JSONResponse(

        status_code=500,

        content={

            "success":
                False,

            "error":
                "Internal server error.",

            "path":
                request.url.path
        }
    )


# ============================================================
# 21. FASTAPI TEST CLIENT
# ============================================================

from fastapi.testclient import TestClient


client = TestClient(
    app
)


print(
    "FastAPI TestClient initialized."
)


# ============================================================
# 22. API TEST — ROOT
# ============================================================

root_response = client.get(
    "/"
)

print("\nROOT STATUS:")
print(
    root_response.status_code
)

print(
    root_response.json()
)


# ============================================================
# 23. API TEST — HEALTH
# ============================================================

health_response = client.get(
    "/health"
)

print("\nHEALTH STATUS:")
print(
    health_response.status_code
)

print(
    health_response.json()
)


# ============================================================
# 24. API TEST — CREATE SESSION
# ============================================================

session_response = client.post(
    "/api/v1/session"
)

print("\nSESSION CREATE STATUS:")
print(
    session_response.status_code
)

session_data = (
    session_response.json()
)

print(
    session_data
)

api_session_id = (
    session_data[
        "session_id"
    ]
)


# ============================================================
# 25. API TEST — CHAT
# ============================================================

chat_payload = {

    "query":
        "How are security vulnerabilities prioritized?",

    "session_id":
        api_session_id
}


chat_response = client.post(

    "/api/v1/chat",

    json=chat_payload
)


print("\nCHAT STATUS:")
print(
    chat_response.status_code
)

chat_data = (
    chat_response.json()
)

print(
    json.dumps(
        chat_data,
        indent=2,
        default=str
    )
)


# ============================================================
# 26. API TEST — FOLLOW-UP QUESTION
# ============================================================
#
# Demonstrates that the same session can be reused.
# ============================================================

follow_up_payload = {

    "query":
        "What information supports that?",

    "session_id":
        api_session_id
}


follow_up_response = client.post(

    "/api/v1/chat",

    json=follow_up_payload
)


print("\nFOLLOW-UP STATUS:")
print(
    follow_up_response.status_code
)

print(
    json.dumps(
        follow_up_response.json(),
        indent=2,
        default=str
    )
)


# ============================================================
# 27. API TEST — SEARCH
# ============================================================

search_payload = {

    "query":
        "How do employees request annual leave?"
}


search_response = client.post(

    "/api/v1/search",

    json=search_payload
)


print("\nSEARCH STATUS:")
print(
    search_response.status_code
)

search_data = (
    search_response.json()
)

print(
    "Confidence:",
    search_data.get(
        "retrieval_confidence"
    )
)

print(
    "Results:",
    search_data.get(
        "final_count"
    )
)


# ============================================================
# 28. API TEST — SESSION HISTORY
# ============================================================

session_history_response = client.get(

    f"/api/v1/session/"
    f"{api_session_id}"
)


print("\nSESSION HISTORY STATUS:")
print(
    session_history_response.status_code
)

session_history = (
    session_history_response.json()
)

print(
    json.dumps(
        session_history,
        indent=2,
        default=str
    )
)


# ============================================================
# 29. API TEST — METRICS
# ============================================================

metrics_response = client.get(
    "/api/v1/metrics"
)

print("\nMETRICS STATUS:")
print(
    metrics_response.status_code
)

print(
    json.dumps(
        metrics_response.json(),
        indent=2,
        default=str
    )
)


# ============================================================
# 30. API TEST — INFO
# ============================================================

info_response = client.get(
    "/api/v1/info"
)

print("\nINFO STATUS:")
print(
    info_response.status_code
)

print(
    json.dumps(
        info_response.json(),
        indent=2,
        default=str
    )
)


# ============================================================
# 31. AUTOMATED API TEST SUITE
# ============================================================

def run_api_test_suite():

    test_results = []

    # --------------------------------------------------------
    # ROOT
    # --------------------------------------------------------

    response = client.get(
        "/"
    )

    test_results.append({

        "test":
            "GET /",

        "status_code":
            response.status_code,

        "passed":
            response.status_code == 200
    })

    # --------------------------------------------------------
    # HEALTH
    # --------------------------------------------------------

    response = client.get(
        "/health"
    )

    test_results.append({

        "test":
            "GET /health",

        "status_code":
            response.status_code,

        "passed":
            response.status_code == 200
    })

    # --------------------------------------------------------
    # CREATE SESSION
    # --------------------------------------------------------

    response = client.post(
        "/api/v1/session"
    )

    session_created = (
        response.status_code == 200
    )

    test_results.append({

        "test":
            "POST /api/v1/session",

        "status_code":
            response.status_code,

        "passed":
            session_created
    })

    if not session_created:

        return pd.DataFrame(
            test_results
        )

    test_session_id = (
        response.json()[
            "session_id"
        ]
    )

    # --------------------------------------------------------
    # CHAT
    # --------------------------------------------------------

    response = client.post(

        "/api/v1/chat",

        json={

            "query":
                "What is the enterprise leave policy?",

            "session_id":
                test_session_id
        }
    )

    test_results.append({

        "test":
            "POST /api/v1/chat",

        "status_code":
            response.status_code,

        "passed":
            response.status_code == 200
    })

    # --------------------------------------------------------
    # SEARCH
    # --------------------------------------------------------

    response = client.post(

        "/api/v1/search",

        json={

            "query":
                "How are security vulnerabilities prioritized?"
        }
    )

    test_results.append({

        "test":
            "POST /api/v1/search",

        "status_code":
            response.status_code,

        "passed":
            response.status_code == 200
    })

    # --------------------------------------------------------
    # SESSION GET
    # --------------------------------------------------------

    response = client.get(

        f"/api/v1/session/"
        f"{test_session_id}"
    )

    test_results.append({

        "test":
            "GET /api/v1/session/{id}",

        "status_code":
            response.status_code,

        "passed":
            response.status_code == 200
    })

    # --------------------------------------------------------
    # METRICS
    # --------------------------------------------------------

    response = client.get(
        "/api/v1/metrics"
    )

    test_results.append({

        "test":
            "GET /api/v1/metrics",

        "status_code":
            response.status_code,

        "passed":
            response.status_code == 200
    })

    # --------------------------------------------------------
    # INFO
    # --------------------------------------------------------

    response = client.get(
        "/api/v1/info"
    )

    test_results.append({

        "test":
            "GET /api/v1/info",

        "status_code":
            response.status_code,

        "passed":
            response.status_code == 200
    })

    # --------------------------------------------------------
    # DELETE SESSION
    # --------------------------------------------------------

    response = client.delete(

        f"/api/v1/session/"
        f"{test_session_id}"
    )

    test_results.append({

        "test":
            "DELETE /api/v1/session/{id}",

        "status_code":
            response.status_code,

        "passed":
            response.status_code == 200
    })

    return pd.DataFrame(
        test_results
    )


api_test_results_df = (
    run_api_test_suite()
)


print("\n============================================================")
print("AUTOMATED API TEST RESULTS")
print("============================================================")

display(
    api_test_results_df
)


# ============================================================
# 32. API TEST SUCCESS RATE
# ============================================================

api_test_success_rate = (
    api_test_results_df[
        "passed"
    ].mean()
)


print(
    "\nAPI TEST SUCCESS RATE:",
    round(
        api_test_success_rate * 100,
        2
    ),
    "%"
)


# ============================================================
# 33. UNKNOWN QUERY SAFETY TEST
# ============================================================

unknown_query = (
    "What is the company's policy "
    "for building cities on Mars?"
)


unknown_response = client.post(

    "/api/v1/chat",

    json={

        "query":
            unknown_query
    }
)


print("\n============================================================")
print("UNKNOWN QUERY SAFETY TEST")
print("============================================================")

print(
    json.dumps(
        unknown_response.json(),
        indent=2,
        default=str
    )
)


# ============================================================
# 34. PERFORMANCE TEST
# ============================================================

performance_queries = [

    "How do employees request annual leave?",

    "How are security vulnerabilities prioritized?",

    "How should business expenses be reimbursed?",

    "What are the enterprise API standards?",

    "How should applications be deployed?"
]


performance_records = []


for query in performance_queries:

    start = time.perf_counter()

    response = client.post(

        "/api/v1/chat",

        json={

            "query":
                query
        }
    )

    latency = (
        time.perf_counter()
        - start
    ) * 1000

    performance_records.append({

        "query":
            query,

        "status_code":
            response.status_code,

        "latency_ms":
            round(
                latency,
                3
            )
    })


performance_df = pd.DataFrame(
    performance_records
)


print("\n============================================================")
print("API PERFORMANCE TEST")
print("============================================================")

display(
    performance_df
)


# ============================================================
# 35. PERFORMANCE SUMMARY
# ============================================================

performance_summary = {

    "average_latency_ms":
        round(
            performance_df[
                "latency_ms"
            ].mean(),
            3
        ),

    "p50_latency_ms":
        round(
            performance_df[
                "latency_ms"
            ].quantile(
                0.50
            ),
            3
        ),

    "p95_latency_ms":
        round(
            performance_df[
                "latency_ms"
            ].quantile(
                0.95
            ),
            3
        ),

    "max_latency_ms":
        round(
            performance_df[
                "latency_ms"
            ].max(),
            3
        ),

    "successful_requests":
        int(
            (
                performance_df[
                    "status_code"
                ] == 200
            ).sum()
        )
}


print("\n============================================================")
print("PERFORMANCE SUMMARY")
print("============================================================")

for key, value in performance_summary.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# 36. RAG QUALITY EVALUATION THROUGH API
# ============================================================

evaluation_queries = [

    "How do employees request annual leave?",

    "How are security vulnerabilities prioritized?",

    "How should business expenses be reimbursed?",

    "What are the enterprise API standards?",

    "How should applications be deployed?",

    "What is the process for insurance claims?"
]


api_evaluation_records = []


for query in evaluation_queries:

    response = client.post(

        "/api/v1/chat",

        json={

            "query":
                query
        }
    )

    data = response.json()

    api_evaluation_records.append({

        "query":
            query,

        "success":
            data.get(
                "success",
                False
            ),

        "grounded":
            data.get(
                "grounded",
                False
            ),

        "confidence":
            data.get(
                "confidence",
                "LOW"
            ),

        "safe_fallback":
            data.get(
                "safe_fallback",
                False
            ),

        "grounding_score":
            data.get(
                "grounding_score",
                0.0
            ),

        "retrieved_chunks":
            data.get(
                "retrieved_chunks",
                0
            ),

        "latency_ms":
            data.get(
                "latency_ms",
                0.0
            ),

        "citation_count":
            len(
                data.get(
                    "citations",
                    []
                )
            )
    })


api_evaluation_df = pd.DataFrame(
    api_evaluation_records
)


print("\n============================================================")
print("API RAG EVALUATION")
print("============================================================")

display(
    api_evaluation_df
)


# ============================================================
# 37. FINAL QUALITY METRICS
# ============================================================

api_rag_success_rate = (
    api_evaluation_df[
        "success"
    ].mean()
)

api_grounding_rate = (
    api_evaluation_df[
        "grounded"
    ].mean()
)

api_fallback_rate = (
    api_evaluation_df[
        "safe_fallback"
    ].mean()
)

api_citation_coverage = (
    (
        api_evaluation_df[
            "citation_count"
        ] > 0
    ).mean()
)

api_average_grounding_score = (
    api_evaluation_df[
        "grounding_score"
    ].mean()
)

api_average_latency = (
    api_evaluation_df[
        "latency_ms"
    ].mean()
)


# ============================================================
# 38. FINAL PART 4 SCORECARD
# ============================================================

PART_4_SCORECARD = {

    "api_test_success_rate_percent":
        round(
            api_test_success_rate * 100,
            2
        ),

    "rag_success_rate_percent":
        round(
            api_rag_success_rate * 100,
            2
        ),

    "grounding_rate_percent":
        round(
            api_grounding_rate * 100,
            2
        ),

    "citation_coverage_percent":
        round(
            api_citation_coverage * 100,
            2
        ),

    "safe_fallback_rate_percent":
        round(
            api_fallback_rate * 100,
            2
        ),

    "average_grounding_score":
        round(
            api_average_grounding_score,
            4
        ),

    "average_api_latency_ms":
        round(
            api_average_latency,
            3
        ),

    "p50_api_latency_ms":
        round(
            metrics.percentile(50),
            3
        ),

    "p95_api_latency_ms":
        round(
            metrics.percentile(95),
            3
        ),

    "p99_api_latency_ms":
        round(
            metrics.percentile(99),
            3
        ),

    "total_api_requests":
        metrics.total_requests
}


print("\n============================================================")
print("DAY 62 PART 4 SCORECARD")
print("============================================================")

for metric, value in PART_4_SCORECARD.items():

    print(
        f"{metric}: {value}"
    )


# ============================================================
# 39. PRODUCTION ENVIRONMENT CONFIGURATION
# ============================================================

ENV_TEMPLATE = """

# ============================================================
# ENTERPRISE AGENTIC RAG ENVIRONMENT
# ============================================================

APP_NAME=enterprise-agentic-rag

APP_ENV=production

APP_VERSION=1.0.0

LOG_LEVEL=INFO

API_HOST=0.0.0.0

API_PORT=8000

TOP_K=5

ENABLE_CITATIONS=true

ENABLE_GROUNDING_VALIDATION=true

ENABLE_SAFE_FALLBACK=true

# Production examples:
#
# VECTOR_DB_URL=
# REDIS_URL=
# DATABASE_URL=
# LLM_API_KEY=
# EMBEDDING_MODEL=
# PROMETHEUS_ENDPOINT=
#
# Never commit secrets to Git.
"""


print("\n============================================================")
print(".ENV TEMPLATE")
print("============================================================")

print(
    ENV_TEMPLATE
)


# ============================================================
# 40. REQUIREMENTS.TXT
# ============================================================

REQUIREMENTS_TXT = """

fastapi
uvicorn[standard]
pydantic
numpy
pandas
scikit-learn
httpx

"""


print("\n============================================================")
print("REQUIREMENTS.TXT")
print("============================================================")

print(
    REQUIREMENTS_TXT
)


# ============================================================
# 41. DOCKERFILE
# ============================================================
#
# This is a deployment template.
# It is not actually building/deploying a container here.
# ============================================================

DOCKERFILE = r'''
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''


print("\n============================================================")
print("DOCKERFILE")
print("============================================================")

print(
    DOCKERFILE
)


# ============================================================
# 42. DOCKERIGNORE
# ============================================================

DOCKERIGNORE = """

__pycache__
*.pyc
.ipynb_checkpoints
.git
.env
venv
.venv
*.log
"""


print("\n============================================================")
print(".DOCKERIGNORE")
print("============================================================")

print(
    DOCKERIGNORE
)


# ============================================================
# 43. PRODUCTION PROJECT STRUCTURE
# ============================================================

PROJECT_STRUCTURE = """
enterprise-agentic-rag/
│
├── app.py
│
├── requirements.txt
├── Dockerfile
├── .dockerignore
├── .env
│
├── config/
│   └── settings.py
│
├── api/
│   ├── chat.py
│   ├── search.py
│   ├── session.py
│   └── health.py
│
├── agents/
│   ├── router.py
│   ├── retrieval_agent.py
│   └── tool_registry.py
│
├── retrieval/
│   ├── vector_search.py
│   ├── keyword_search.py
│   └── reranker.py
│
├── rag/
│   ├── context_builder.py
│   ├── generator.py
│   ├── grounding.py
│   └── citations.py
│
├── memory/
│   └── session_manager.py
│
├── monitoring/
│   ├── metrics.py
│   └── logging.py
│
├── tests/
│   ├── test_api.py
│   ├── test_retrieval.py
│   └── test_rag.py
│
└── README.md
"""


print("\n============================================================")
print("PRODUCTION PROJECT STRUCTURE")
print("============================================================")

print(
    PROJECT_STRUCTURE
)


# ============================================================
# 44. PRODUCTION ARCHITECTURE
# ============================================================

PRODUCTION_ARCHITECTURE = """
                    ENTERPRISE USERS
                           │
                           ▼
                    API GATEWAY / WAF
                           │
                           ▼
                     LOAD BALANCER
                           │
                           ▼
                    FASTAPI SERVICE
                           │
              ┌────────────┴────────────┐
              │                         │
              ▼                         ▼
        SESSION MEMORY            AGENT ROUTER
        Redis / DB                    │
                                      ▼
                              QUERY TRANSFORMATION
                                      │
                                      ▼
                               HYBRID RETRIEVAL
                              ┌────────┴────────┐
                              │                 │
                              ▼                 ▼
                         VECTOR DB        KEYWORD INDEX
                              │                 │
                              └────────┬────────┘
                                       ▼
                                   RERANKER
                                       │
                                       ▼
                                  TOP-K CONTEXT
                                       │
                                       ▼
                               ENTERPRISE LLM
                                       │
                                       ▼
                              GROUNDING VALIDATOR
                                       │
                                       ▼
                                CITATION MANAGER
                                       │
                                       ▼
                                  SAFETY GATE
                                       │
                                       ▼
                                FINAL RESPONSE
                                       │
                 ┌─────────────────────┴──────────────────┐
                 │                                        │
                 ▼                                        ▼
             MONITORING                               LOGGING
          Prometheus/Grafana                      Centralized Logs
"""


print("\n============================================================")
print("PRODUCTION ARCHITECTURE")
print("============================================================")

print(
    PRODUCTION_ARCHITECTURE
)


# ============================================================
# 45. MONITORING CHECKLIST
# ============================================================

MONITORING_CHECKLIST = {

    "API": [

        "Request count",

        "Success rate",

        "Error rate",

        "Average latency",

        "P50 latency",

        "P95 latency",

        "P99 latency"
    ],

    "Retrieval": [

        "Candidate count",

        "Top-K relevance",

        "Retrieval confidence",

        "Empty retrieval rate",

        "Reranking quality",

        "Hit@K",

        "MRR"
    ],

    "RAG": [

        "Grounding rate",

        "Grounding score",

        "Citation coverage",

        "Answer relevance",

        "Safe fallback rate",

        "Hallucination rate",

        "No-answer rate"
    ],

    "Infrastructure": [

        "CPU usage",

        "Memory usage",

        "Container health",

        "Database health",

        "Vector DB health",

        "LLM availability"
    ]
}


print("\n============================================================")
print("PRODUCTION MONITORING CHECKLIST")
print("============================================================")

for category, items in MONITORING_CHECKLIST.items():

    print(
        f"\n{category}"
    )

    for item in items:

        print(
            "  ✓",
            item
        )


# ============================================================
# 46. SECURITY CHECKLIST
# ============================================================

SECURITY_CHECKLIST = [

    "Authentication",

    "Authorization",

    "Role-Based Access Control",

    "Document-Level Access Control",

    "API Rate Limiting",

    "Input Validation",

    "Prompt Injection Protection",

    "PII Protection",

    "Secrets Management",

    "Encryption In Transit",

    "Encryption At Rest",

    "Audit Logging",

    "Dependency Scanning",

    "Container Security",

    "Network Isolation"
]


print("\n============================================================")
print("SECURITY CHECKLIST")
print("============================================================")

for item in SECURITY_CHECKLIST:

    print(
        "✓",
        item
    )


# ============================================================
# 47. FINAL END-TO-END DEMO
# ============================================================

final_demo_queries = [

    "How do employees request annual leave?",

    "How are security vulnerabilities prioritized?",

    "How should business expenses be reimbursed?"
]


final_demo_records = []


print("\n============================================================")
print("FINAL END-TO-END DEMO")
print("============================================================")


for query in final_demo_queries:

    start = time.perf_counter()

    response = client.post(

        "/api/v1/chat",

        json={

            "query":
                query
        }
    )

    latency = (
        time.perf_counter()
        - start
    ) * 1000

    data = response.json()

    print("\n------------------------------------------------------------")

    print(
        "USER:",
        query
    )

    print(
        "\nAI:",
        data.get(
            "answer"
        )
    )

    print(
        "\nConfidence:",
        data.get(
            "confidence"
        )
    )

    print(
        "Grounded:",
        data.get(
            "grounded"
        )
    )

    print(
        "Citations:",
        len(
            data.get(
                "citations",
                []
            )
        )
    )

    print(
        "Latency:",
        round(
            latency,
            3
        ),
        "ms"
    )

    final_demo_records.append({

        "query":
            query,

        "success":
            data.get(
                "success",
                False
            ),

        "grounded":
            data.get(
                "grounded",
                False
            ),

        "confidence":
            data.get(
                "confidence",
                "LOW"
            ),

        "citations":
            len(
                data.get(
                    "citations",
                    []
                )
            ),

        "latency_ms":
            round(
                latency,
                3
            )
    })


final_demo_df = pd.DataFrame(
    final_demo_records
)


# ============================================================
# 48. FINAL SERVICE METRICS
# ============================================================

print("\n============================================================")
print("FINAL SERVICE METRICS")
print("============================================================")

final_service_metrics = (
    metrics.snapshot()
)


for key, value in final_service_metrics.items():

    print(
        f"{key}: {value}"
    )


# ============================================================
# 49. FINAL VALIDATION
# ============================================================

part_4_validation = {

    "FastAPI application created":
        app is not None,

    "Root endpoint":
        root_response.status_code == 200,

    "Health endpoint":
        health_response.status_code == 200,

    "Session creation":
        session_response.status_code == 200,

    "Chat endpoint":
        chat_response.status_code == 200,

    "Follow-up endpoint":
        follow_up_response.status_code == 200,

    "Search endpoint":
        search_response.status_code == 200,

    "Session history":
        session_history_response.status_code == 200,

    "Metrics endpoint":
        metrics_response.status_code == 200,

    "Info endpoint":
        info_response.status_code == 200,

    "Automated API tests":
        api_test_success_rate == 1.0,

    "RAG evaluation":
        len(
            api_evaluation_df
        ) > 0,

    "Performance evaluation":
        len(
            performance_df
        ) > 0,

    "Docker configuration":
        "FROM python" in DOCKERFILE,

    "Production architecture":
        "FASTAPI SERVICE"
        in PRODUCTION_ARCHITECTURE,

    "Security checklist":
        len(
            SECURITY_CHECKLIST
        ) > 0,

    "Monitoring checklist":
        len(
            MONITORING_CHECKLIST
        ) > 0
}


print("\n============================================================")
print("PART 4 VALIDATION")
print("============================================================")


all_part_4_passed = True


for check, status in part_4_validation.items():

    print(
        f"{'PASS' if status else 'FAIL'} | {check}"
    )

    if not status:

        all_part_4_passed = False


print("\nOverall Part 4 Status:")


if all_part_4_passed:

    print(
        "✅ PART 4 COMPLETED SUCCESSFULLY"
    )

else:

    print(
        "⚠️ PART 4 REQUIRES ATTENTION"
    )


# ============================================================
# 50. IMPORTANT VARIABLES CREATED
# ============================================================

print("\n============================================================")
print("IMPORTANT VARIABLES CREATED IN PART 4")
print("============================================================")


PART_4_VARIABLES = [

    "PRODUCTION_CONFIG",

    "ApplicationMetrics",

    "metrics",

    "SESSION_STORE",

    "create_session",

    "get_session",

    "delete_session",

    "add_session_message",

    "build_session_context",

    "ChatRequest",

    "ChatResponse",

    "SearchRequest",

    "SessionCreateResponse",

    "HealthResponse",

    "app",

    "client",

    "api_session_id",

    "api_test_results_df",

    "api_test_success_rate",

    "performance_df",

    "performance_summary",

    "api_evaluation_df",

    "PART_4_SCORECARD",

    "ENV_TEMPLATE",

    "REQUIREMENTS_TXT",

    "DOCKERFILE",

    "DOCKERIGNORE",

    "PROJECT_STRUCTURE",

    "PRODUCTION_ARCHITECTURE",

    "MONITORING_CHECKLIST",

    "SECURITY_CHECKLIST",

    "final_demo_df",

    "final_service_metrics",

    "part_4_validation"
]


for variable in PART_4_VARIABLES:

    print(
        "→",
        variable
    )


# ============================================================
# 51. COMPLETE DAY 62 ARCHITECTURE
# ============================================================

print("""
============================================================
DAY 62/100 — COMPLETE ENTERPRISE AGENTIC RAG SYSTEM
============================================================


                         USER
                           │
                           ▼
                    ┌─────────────┐
                    │   FastAPI   │
                    └──────┬──────┘
                           │
                           ▼
                  Request Validation
                           │
                           ▼
                  Session Management
                           │
                           ▼
                 Agentic Orchestrator
                           │
             ┌─────────────┴─────────────┐
             │                           │
             ▼                           ▼
      Query Understanding          Tool / Route Selection
             │                           │
             └─────────────┬─────────────┘
                           ▼
                  Query Transformation
                           │
                           ▼
                    Query Expansion
                           │
                           ▼
                 ┌─────────────────┐
                 │   Retrieval     │
                 └────────┬────────┘
                          │
              ┌───────────┴───────────┐
              │                       │
              ▼                       ▼
         Vector Search          Keyword Search
              │                       │
              └───────────┬───────────┘
                          ▼
                    Score Fusion
                          │
                          ▼
                      Reranking
                          │
                          ▼
                        Top-K
                          │
                          ▼
               Retrieval Confidence
                          │
                          ▼
                Context Construction
                          │
                          ▼
                 Grounded Generation
                          │
                          ▼
                Grounding Validation
                          │
                    ┌─────┴─────┐
                    │           │
                   PASS         FAIL
                    │           │
                    ▼           ▼
                Citations   Safe Fallback
                    │
                    ▼
               Final Response
                    │
          ┌─────────┴─────────┐
          │                   │
          ▼                   ▼
      Monitoring           Logging
          │                   │
          ▼                   ▼
    Prometheus/Grafana    Central Logs


============================================================
PART 1
============================================================

Enterprise Documents
        ↓
Cleaning
        ↓
Chunking
        ↓
Metadata
        ↓
Vector Representation
        ↓
Vector Database
        ↓
Similarity Search


============================================================
PART 2
============================================================

User Query
        ↓
Query Understanding
        ↓
Agent Routing
        ↓
Query Expansion
        ↓
Metadata Filtering
        ↓
Candidate Retrieval
        ↓
Reranking
        ↓
Retrieval Confidence


============================================================
PART 3
============================================================

Top-K Results
        ↓
Context Construction
        ↓
Grounded Generation
        ↓
Claim Extraction
        ↓
Grounding Validation
        ↓
Source Citations
        ↓
Safe Fallback


============================================================
PART 4
============================================================

FastAPI
        ↓
Session Management
        ↓
REST APIs
        ↓
Monitoring
        ↓
Testing
        ↓
Performance Evaluation
        ↓
Docker
        ↓
Cloud Architecture
        ↓
Security
        ↓
Production Readiness


============================================================
DAY 62 COMPLETE
============================================================
""")


# ============================================================
# 52. FINAL INTERVIEW-READY EXPLANATION
# ============================================================

print("""
============================================================
DAY 62 — INTERVIEW EXPLANATION
============================================================

"I built an enterprise Agentic RAG system that combines vector
retrieval, agentic query processing, reranking, grounded
generation and production APIs.

I started by creating an enterprise document retrieval layer
where documents are cleaned, chunked, enriched with metadata
and represented in a vector-searchable structure.

Then I added an agentic retrieval layer. The system analyzes
the user's query, identifies the relevant domain, performs
query transformation and expansion, applies metadata filters,
retrieves candidate chunks and reranks them before calculating
retrieval confidence.

The next layer is RAG. The top-ranked chunks are converted into
a structured context and passed to the generation layer. I
added grounding validation and claim-level validation so that
the generated response can be checked against retrieved
evidence. I also added source citations and a safe fallback
when sufficient evidence is unavailable.

Finally, I exposed the complete pipeline through FastAPI.
The service provides chat, search, session, health, metrics and
information APIs. I implemented request IDs, latency tracking,
success/error monitoring, session management and automated API
testing.

I also evaluated API latency, grounding rate, citation
coverage, RAG success rate and safe fallback behavior.

For deployment, I created Docker and environment configuration
templates and designed a cloud-ready architecture with an API
gateway, load balancer, FastAPI services, vector database,
session store, enterprise LLM, grounding validation and
monitoring.

The architecture is model-independent, so the lightweight
development implementation can later be replaced with
production embedding models, managed vector databases,
enterprise LLMs, Redis and Prometheus/Grafana without changing
the core application design."


============================================================
KEY INTERVIEW FLOW
============================================================

Problem
   ↓
Document Ingestion
   ↓
Vector Database
   ↓
Agentic Retrieval
   ↓
Query Expansion
   ↓
Metadata Filtering
   ↓
Reranking
   ↓
Retrieval Confidence
   ↓
Context Construction
   ↓
RAG
   ↓
Grounding
   ↓
Citations
   ↓
Safe Fallback
   ↓
FastAPI
   ↓
Session Management
   ↓
Monitoring
   ↓
Testing
   ↓
Docker
   ↓
Cloud Architecture


============================================================
DAY 62 FINAL
============================================================
""")

2026-09-29 04:46:47,969 | INFO | Enterprise Agentic RAG service initialized.
2026-09-29 04:46:48,134 | INFO | request_id=e3a4ab61-3f0e-4564-b56a-c8f74e984e44 method=GET path=/ latency_ms=1.844 success=True
2026-09-29 04:46:48,137 | INFO | HTTP Request: GET http://testserver/ "HTTP/1.1 200 OK"


Part 4 libraries loaded successfully.
✅ Part 3 agentic_rag() detected.

PRODUCTION CONFIGURATION
application_name: Enterprise Agentic RAG Assistant
version: 1.0.0
environment: development
api_prefix: /api/v1
default_top_k: 5
max_query_length: 1000
max_session_history: 10
request_timeout_seconds: 30
enable_citations: True
enable_grounding_validation: True
enable_safe_fallback: True
monitoring_enabled: True
FastAPI TestClient initialized.

ROOT STATUS:
200
{'service': 'Enterprise Agentic RAG Assistant', 'version': '1.0.0', 'status': 'running', 'message': 'Enterprise Agentic RAG API'}


2026-09-29 04:46:48,143 | INFO | request_id=400b08b2-72dd-403a-9655-8e7a92403b1e method=GET path=/health latency_ms=2.710 success=True
2026-09-29 04:46:48,147 | INFO | HTTP Request: GET http://testserver/health "HTTP/1.1 200 OK"
2026-09-29 04:46:48,153 | INFO | request_id=77ae1540-b1c2-4f89-aacd-20be7f331a8b method=POST path=/api/v1/session latency_ms=2.109 success=True
2026-09-29 04:46:48,175 | INFO | HTTP Request: POST http://testserver/api/v1/session "HTTP/1.1 200 OK"
2026-09-29 04:46:48,205 | INFO | request_id=a6bee33f-afb8-45bc-af4f-dbfe813b85d9 method=POST path=/api/v1/chat latency_ms=25.448 success=True
2026-09-29 04:46:48,210 | INFO | HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
2026-09-29 04:46:48,227 | INFO | request_id=dea1c3fc-cbb5-4ac1-bc72-1d9744fbe48d method=POST path=/api/v1/chat latency_ms=12.849 success=True
2026-09-29 04:46:48,230 | INFO | HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
2026-09-29 04:46:48,261 | INFO | request


HEALTH STATUS:
200
{'status': 'healthy', 'service': 'Enterprise Agentic RAG Assistant', 'version': '1.0.0', 'timestamp': '2026-09-29T04:46:48.142200+00:00'}

SESSION CREATE STATUS:
200
{'session_id': 'b9d53400-466c-49c7-a74c-d32b8e566c23', 'created_at': '2026-09-29T04:46:48.153244+00:00'}

CHAT STATUS:
200
{
  "request_id": "3cdb939d-9049-4ec5-8f0e-fddfecdddb48",
  "session_id": "b9d53400-466c-49c7-a74c-d32b8e566c23",
  "query": "How are security vulnerabilities prioritized?",
  "answer": "Security vulnerabilities should be identified, assessed, prioritized, and remediated according to organizational risk management procedures. Critical vulnerabilities require expedited remediation. Employees must protect confidential company information and follow approved security controls. Security incidents, suspicious activity, and potential data exposure must be reported through the approved security process.\n\nSources: SOURCE 1, SOURCE 2",
  "success": true,
  "confidence": "MEDIUM",
  "ground

2026-09-29 04:46:48,340 | INFO | HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
2026-09-29 04:46:48,366 | INFO | request_id=a347e27e-6c7c-4d29-b88b-a2dee582f4c6 method=POST path=/api/v1/search latency_ms=23.210 success=True
2026-09-29 04:46:48,368 | INFO | HTTP Request: POST http://testserver/api/v1/search "HTTP/1.1 200 OK"
2026-09-29 04:46:48,375 | INFO | request_id=9732edd4-0764-493a-b049-057ec8cb27f9 method=GET path=/api/v1/session/a3a1afb6-3407-4f18-b9a9-ee670c912ef9 latency_ms=1.225 success=True
2026-09-29 04:46:48,378 | INFO | HTTP Request: GET http://testserver/api/v1/session/a3a1afb6-3407-4f18-b9a9-ee670c912ef9 "HTTP/1.1 200 OK"
2026-09-29 04:46:48,382 | INFO | request_id=04408ef9-6b04-4502-92d9-3bf556b88af7 method=GET path=/api/v1/metrics latency_ms=1.811 success=True
2026-09-29 04:46:48,385 | INFO | HTTP Request: GET http://testserver/api/v1/metrics "HTTP/1.1 200 OK"
2026-09-29 04:46:48,389 | INFO | request_id=d566b796-0e8f-41f4-824c-dd35fe7fabf3 method=GE

...


API RAG EVALUATION


,query,success,grounded,confidence,safe_fallback,grounding_score,retrieved_chunks,latency_ms,citation_count
0,How do employees request annual leave?,True,True,MEDIUM,False,0.9730,2,20.896,2
1,How are security vulnerabilities prioritized?,True,True,MEDIUM,False,0.9706,2,20.780,2
2,How should business expenses be reimbursed?,True,True,MEDIUM,False,0.9655,2,20.910,2
3,What are the enterprise API standards?,True,True,MEDIUM,False,0.9500,2,20.951,2
4,How should applications be deployed?,True,True,MEDIUM,False,0.8571,2,9.236,2
5,What is the process for insurance claims?,True,True,MEDIUM,False,0.9688,2,20.999,2



DAY 62 PART 4 SCORECARD
api_test_success_rate_percent: 100.0
rag_success_rate_percent: 100.0
grounding_rate_percent: 100.0
citation_coverage_percent: 100.0
safe_fallback_rate_percent: 0.0
average_grounding_score: 0.9475
average_api_latency_ms: 18.962
p50_api_latency_ms: 12.395
p95_api_latency_ms: 24.84
p99_api_latency_ms: 26.384
total_api_requests: 30

.ENV TEMPLATE


# ============================================================
# ENTERPRISE AGENTIC RAG ENVIRONMENT
# ============================================================

APP_NAME=enterprise-agentic-rag

APP_ENV=production

APP_VERSION=1.0.0

LOG_LEVEL=INFO

API_HOST=0.0.0.0

API_PORT=8000

TOP_K=5

ENABLE_CITATIONS=true

ENABLE_GROUNDING_VALIDATION=true

ENABLE_SAFE_FALLBACK=true

# Production examples:
#
# VECTOR_DB_URL=
# REDIS_URL=
# DATABASE_URL=
# LLM_API_KEY=
# EMBEDDING_MODEL=
# PROMETHEUS_ENDPOINT=
#
# Never commit secrets to Git.


REQUIREMENTS.TXT


fastapi
uvicorn[standard]
pydantic
numpy
pandas
scikit-learn
httpx



2026-09-29 04:46:48,766 | INFO | request_id=f8418294-2cb3-44f1-8629-0b7162fdc5b6 method=POST path=/api/v1/chat latency_ms=23.117 success=True
2026-09-29 04:46:48,770 | INFO | HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
2026-09-29 04:46:48,797 | INFO | request_id=325bdb01-b063-44f0-8aa0-ebdfe154e1a0 method=POST path=/api/v1/chat latency_ms=22.953 success=True
2026-09-29 04:46:48,800 | INFO | HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"
2026-09-29 04:46:48,826 | INFO | request_id=070ff9a7-b703-4a67-8bcb-ac3971155bb1 method=POST path=/api/v1/chat latency_ms=22.436 success=True
2026-09-29 04:46:48,829 | INFO | HTTP Request: POST http://testserver/api/v1/chat "HTTP/1.1 200 OK"



------------------------------------------------------------
USER: How do employees request annual leave?

AI: Employees are eligible for annual leave according to their employment category and organizational policy. Employees should review the official HR policy for applicable leave balances and eligibility requirements. Leave requests should be submitted through the employee management portal before the planned absence. Managers are responsible for reviewing and approving leave requests. Emergency leave should be communicated to the reporting manager as soon as reasonably possible.

Sources: SOURCE 1

Confidence: MEDIUM
Grounded: True
Citations: 2
Latency: 30.751 ms

------------------------------------------------------------
USER: How are security vulnerabilities prioritized?

AI: Security vulnerabilities should be identified, assessed, prioritized, and remediated according to organizational risk management procedures. Critical vulnerabilities require expedited remediation. Employ